# X5_Q2 — Core descriptivo (Pearson, Spearman, coseno, regímenes, estabilidad, export)

Segundo notebook de **X5_alt_v2** (ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2.

Consume la tabla de estado que construye `X5_Q1.ipynb`
(`resources/x5_alt/exp_Q1/{valor}_tabla_estado_v2.csv`: precio + X3 + X2 + variables
de drawdown/volatilidad/liquidez/fundamentales/ruido/tendencia) y agrega
sobre ella el análisis descriptivo variable↔precio — correlación clásica
(Pearson/Spearman), coseno `cos(Pv,Xv)`, no linealidades, ceteris paribus,
redundancia, regímenes y estabilidad — más el export estructurado de
métricas que retoma `X5_Q3.ipynb`.

Mismo input por lista `valores` que `X5_Q1.ipynb` (no el `valor` singular de v1).

In [ ]:
import sys
from pathlib import Path

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

valores = list(cfg.VALORES)  # default = todos los activos de config.py; recortar a mano acá para un análisis puntual con menos

## 0. Preparación

Imports y configuración común al resto del notebook: `config.py` (rutas de `resources/x5_alt/`) y las opciones de despliegue de `pandas` usadas en los DataFrames anchos de las secciones siguientes.

In [ ]:
import numpy as np
import pandas as pd

from x5_alt_export import carpeta_exp, limpiar_exp, exportar_zip  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

CARPETA_EXP = limpiar_exp('X5_Q2')  # limpia resources/x5_alt/exp_Q2 al inicio de cada corrida

print(f"Activos: {valores}")

## 1. Carga de la tabla de estado de X5_Q1 (Q2-1)

Lee `{valor}_tabla_estado_v2.csv` por cada activo de `valores` — generada
por `X5_Q1.ipynb` (Sección 10), no recalculada acá. Si no existe, correr
`X5_Q1.ipynb` primero.

In [3]:
def _cargar_tabla_estado(valor):
    path = carpeta_exp('X5_Q1') / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

BTCUSD: 40283 filas x 72 columnas — 2021-10-27 16:00:00 → 2026-06-02 21:00:00


## 2. Gráficos a doble eje precio + variable (Q2-2)

**Objetivo**: inspeccionar visualmente el co-movimiento entre el precio (`Close`) y
cada variable de estado de la tabla, conservando la escala nativa de cada una en
vez de forzarlas a una sola unidad.

**Pregunta**: ¿la variable se mueve en sincronía, con rezago o de forma
independiente respecto al precio, vista en su propia escala?

**Metodología**: por cada variable de la tabla (todas las columnas salvo las de
precio/volumen crudo — `DateTime`, `Open`, `High`, `Low`, `Close`, `Tick_Volume`,
`Spread`, `Real_Volume`), un gráfico de doble eje: precio en el eje izquierdo,
variable en el eje derecho, con **ambos ejes centrados en su propia media** (mismo
punto medio visual) para que una desviación de la variable respecto a su media sea
comparable a simple vista con una desviación del precio respecto a la suya, sin
que la diferencia de unidades/magnitud lo distorsione. Para las variables
categóricas (`regimen_tendencia`, `ruptura_rango`, `fundamental_direccion_cambio`)
el eje doble no es interpretable como línea continua en su escala nativa — ahí se
usa el caso al que se refiere el TO DO como "series normalizadas superpuestas
cuando las escalas sean incompatibles": precio y variable, ambas como z-score, en
un solo eje.

**Interpretación**: exploración visual únicamente — adelanta intuición sobre qué
variables acompañan o se desacoplan del precio, pero la cuantificación
(Pearson/Spearman/coseno) es responsabilidad de Q2-3/Q2-4, no de este bloque.

**Limitaciones**: con variables 100% nulas en el snapshot actual de Mac (`x2_score`
y sus derivadas para BTCUSD, ver catálogo de Q1 — sin solapamiento temporal con
datos reales de X2) no hay superposición no nula con el precio — el gráfico se
omite y se imprime un aviso en vez de generar un panel vacío.

In [4]:
import time

import matplotlib.pyplot as plt

EXCLUIR_PRECIO = {'DateTime', 'Open', 'High', 'Low', 'Close', 'Tick_Volume', 'Spread', 'Real_Volume'}
CATEGORICAS = {'regimen_tendencia', 'ruptura_rango', 'fundamental_direccion_cambio'}
PRECIO_COL = 'Close'


def _progreso(contador, total, t_inicio):
    """Imprime avance {contador}/{total} y ETA estimado (min), por tiempo promedio de los gráficos ya generados en este bloque."""
    transcurrido = time.time() - t_inicio
    eta_min = (transcurrido / contador) * (total - contador) / 60
    print(f"Gráfico {contador}/{total} — ETA estimado: {eta_min:.1f} min")


def _ylim_centrado(serie, k=1.05):
    """Límites del eje centrados en la media de `serie`, con margen simétrico según su rango real."""
    media = serie.mean()
    margen = max(serie.max() - media, media - serie.min()) * k
    if not np.isfinite(margen) or margen == 0:
        margen = 1.0
    return media - margen, media + margen


def _serie_numerica(serie):
    """Codifica categóricas (ej. regimen_tendencia) a enteros para poder graficarlas/normalizarlas."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    codigos, _ = pd.factorize(serie)
    return pd.Series(codigos, index=serie.index).replace(-1, np.nan)


def _zscore(serie):
    std = serie.std()
    if not std or not np.isfinite(std):
        return serie * 0
    return (serie - serie.mean()) / std


def plot_doble_eje(df, variable, valor, precio_col=PRECIO_COL):
    """Q2-2: precio (eje izq.) vs. variable (eje der.), ambos centrados en su propia media.
    Si `variable` es categórica, usa un solo eje con ambas series normalizadas (z-score)."""
    serie_variable = _serie_numerica(df[variable])
    sub = pd.DataFrame({'DateTime': df['DateTime'], precio_col: df[precio_col], variable: serie_variable}).dropna()
    if sub.empty:
        print(f"{variable}: sin superposición temporal no nula con {precio_col} — se omite.")
        return False

    fig, ax1 = plt.subplots(figsize=(14, 4))

    if variable in CATEGORICAS:
        ax1.plot(sub['DateTime'], _zscore(sub[precio_col]), color='#1f77b4', linewidth=0.8, label=precio_col)
        ax1.plot(sub['DateTime'], _zscore(sub[variable]), color='#d62728', linewidth=0.8, alpha=0.8, label=variable)
        ax1.set_ylabel('z-score')
        ax1.set_title(f"{valor} — {precio_col} vs. {variable} — normalizadas superpuestas (categórica, eje doble no aplica)")
        ax1.legend(fontsize=8)
    else:
        ax1.plot(sub['DateTime'], sub[precio_col], color='#1f77b4', linewidth=0.8)
        ax1.set_ylabel(precio_col, color='#1f77b4')
        ax1.tick_params(axis='y', labelcolor='#1f77b4')
        ax1.set_ylim(_ylim_centrado(sub[precio_col]))

        ax2 = ax1.twinx()
        ax2.plot(sub['DateTime'], sub[variable], color='#d62728', linewidth=0.8, alpha=0.85)
        ax2.set_ylabel(variable, color='#d62728')
        ax2.tick_params(axis='y', labelcolor='#d62728')
        ax2.set_ylim(_ylim_centrado(sub[variable]))

        ax1.set_title(f"{valor} — {precio_col} (izq.) vs. {variable} (der.) — ejes centrados en su propia media")

    ax1.set_xlabel('DateTime')
    plt.tight_layout()
    plt.show()
    return True

In [5]:
total_graficos_q2_2 = sum(len([c for c in tabla.columns if c not in EXCLUIR_PRECIO]) for tabla in tablas.values())
t_inicio_q2_2 = time.time()
contador_q2_2 = 0

for valor, tabla in tablas.items():
    variables_q2_2 = [c for c in tabla.columns if c not in EXCLUIR_PRECIO]
    graficados = 0
    for variable in variables_q2_2:
        if plot_doble_eje(tabla, variable, valor):
            graficados += 1
        contador_q2_2 += 1
        _progreso(contador_q2_2, total_graficos_q2_2, t_inicio_q2_2)
    print(f"{valor}: {graficados}/{len(variables_q2_2)} variables graficadas (resto omitido por falta de datos no nulos).")

x2_score: sin superposición temporal no nula con Close — se omite.
x2_score_cross: sin superposición temporal no nula con Close — se omite.
x2_score_tendencia: sin superposición temporal no nula con Close — se omite.


fundamental_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_direccion_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_magnitud_cambio: sin superposición temporal no nula con Close — se omite.


fundamental_estabilidad_90d: sin superposición temporal no nula con Close — se omite.
fundamental_contradiccion: sin superposición temporal no nula con Close — se omite.


BTCUSD: 56/64 variables graficadas (resto omitido por falta de datos no nulos).


## 3. Núcleo clásico variable↔precio (Q2-3)

**Objetivo**: cuantificar la asociación contemporánea entre cada variable de
estado de Q1 y el comportamiento del activo en el mismo instante — el bloque
heredado de v1 (`scripts/otros_strings/X5_P2.ipynb` §3), ahora aplicado sobre
las variables de estado de Q1 en vez de los indicadores crudos de v1.

**Pregunta**: ¿con qué fuerza y en qué dirección se mueve cada variable junto
al precio, en el mismo instante `t`?

**Decisión de diseño — dos paneles, no uno**: el documento rector
(`x5_contexto_conversacion.md` §15) advierte que centrar el análisis en el
*nivel* del precio (`Close`) es un error ya detectado en v1 — `Close` no es
estacionario (BTCUSD recorrió ~20.000→~100.000 USD entre 2021 y 2026), así que
casi cualquier variable con tendencia propia termina "correlacionada" con el
nivel del precio por coincidencia temporal, no por una relación informativa.
Por eso cada variable se evalúa contra **dos objetivos**:

- **Panel A — nivel (`Close`), referencial**: fiel al texto original del
  ítem/plan ("variable↔precio"), se muestra pero se marca expresamente como
  no concluyente por la no estacionariedad señalada arriba.
- **Panel B — retorno logarítmico contemporáneo
  (`retorno_log_1h = log(Close_t / Close_{t-1})`), principal**: estacionario,
  es el que alimenta el ranking de esta sección y los ítems posteriores
  (`Q2-5` export, `Q2-11` síntesis).

**Metodología** (por variable, por panel): Pearson (relación lineal),
Spearman (relación monótona), scatter + regresión lineal simple (descriptiva,
no predictiva), y objetivo medio por decil de la variable (cuantiles/bins) —
las mismas métricas/gráficos que usaba v1, con la misma convención de
dirección/fuerza (\|r\| < 0.10 nula/despreciable, < 0.30 débil, < 0.50
moderada, < 0.70 fuerte, ≥ 0.70 muy fuerte, usando Pearson como referencia).

**Variables analizadas**: todas las de la Sección 2 salvo `regimen_tendencia`
— nominal sin orden natural (bull/bear/sideways): forzar Pearson/Spearman
sobre un `factorize` arbitrario de esas 3 categorías no sería interpretable,
así que se trata aparte con una comparación directa de medias por categoría
(sin asumir orden).

**Interpretación**: un Pearson bajo contra el *nivel* no implica que la
variable sea inútil — un oscilador acotado 0-100 como el RSI no tiene por qué
moverse con el nivel de precio (BTC a 20.000 o a 100.000 puede tener RSI=50
igual). Es la relación contra el *retorno* (Panel B) la que importa para
pensar configuración. Este bloque es contemporáneo (`X_t` vs.
`retorno_t`/`Close_t`) — no implica capacidad predictiva sobre el futuro, eso
es la Parte II (fuera de esta sección).

**Limitaciones**: correlación no implica causalidad; con ~40.000
observaciones horarias fuertemente autocorrelacionadas (no independientes),
los p-values salen extremadamente pequeños incluso para relaciones triviales
— se muestran solo como referencia bajo el supuesto simplificado de
independencia, no deben leerse literalmente; no cubre todavía no linealidades
más allá de lo que revela el decil (`Q2-7`), ceteris paribus (`Q2-6`) ni
estabilidad temporal (`Q2-9`) — ítems separados y posteriores; variables con
menos de 30 observaciones no nulas (fundamentales 100% NaN en este snapshot
de Mac) se omiten del loop con aviso, igual que en la Sección 2.

In [6]:
from scipy import stats as scipy_stats

N_BINS = 10
REGIMEN_NOMINAL = 'regimen_tendencia'  # sin orden natural (bull/bear/sideways) — tratado aparte, no en el loop numérico
TARGETS_Q2_3 = ['Close', 'retorno_log_1h']  # Panel A (referencial) y Panel B (principal)
TARGETS_DERIVADOS = {'retorno_log_1h'}  # columnas de retorno calculadas en este notebook, nunca variables explicativas — Q2-8 agrega los retorno_pasado_*


def interpretar_fuerza(r):
    """Etiqueta dirección/fuerza de Pearson — misma convención práctica heredada de v1 (otros_strings/X5_P2.ipynb)."""
    signo = 'positiva' if r >= 0 else 'negativa'
    a = abs(r)
    if a < 0.10:
        fuerza = 'nula/despreciable'
    elif a < 0.30:
        fuerza = 'débil'
    elif a < 0.50:
        fuerza = 'moderada'
    elif a < 0.70:
        fuerza = 'fuerte'
    else:
        fuerza = 'muy fuerte'
    return f"{signo} {fuerza}"


for tabla in tablas.values():
    tabla['retorno_log_1h'] = np.log(tabla['Close'] / tabla['Close'].shift(1))  # estacionario, Panel B


def _variables_analisis_q2_3(tabla):
    return [c for c in tabla.columns if c not in EXCLUIR_PRECIO and c != REGIMEN_NOMINAL and c not in TARGETS_DERIVADOS]


print(f"Variables a evaluar en Q2-3 (BTCUSD): {len(_variables_analisis_q2_3(tablas['BTCUSD']))} "
      f"(excluida {REGIMEN_NOMINAL}, tratada aparte más abajo)")

Variables a evaluar en Q2-3 (BTCUSD): 63 (excluida regimen_tendencia, tratada aparte más abajo)


In [7]:
def analizar_variable_vs_target(df, col, target_col, valor, n_bins=N_BINS):
    """Q2-3: Pearson/Spearman + scatter/regresión + {target_col} medio por decil de `col`."""
    sub = df[[col, target_col]].dropna()
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None

    pear_r, pear_p = scipy_stats.pearsonr(x, y)
    spear_r, spear_p = scipy_stats.spearmanr(x, y)

    try:
        bins = pd.qcut(sub[col], n_bins, duplicates='drop')
    except ValueError:
        bins = pd.qcut(sub[col].rank(method='first'), n_bins)
    por_decil = sub.groupby(bins, observed=True)[target_col].mean()
    por_decil.index = [f"b{i + 1}" for i in range(len(por_decil))]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"{valor} — {col} vs. {target_col} — Pearson={pear_r:.3f} Spearman={spear_r:.3f} ({interpretar_fuerza(pear_r)})",
                 fontsize=10)

    ax = axes[0]
    ax.scatter(x, y, s=2, alpha=0.15, color='#1f77b4')
    coef = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='#d62728', linewidth=1.5)
    ax.set_xlabel(col)
    ax.set_ylabel(target_col)
    ax.set_title('Scatter + regresión lineal')

    ax = axes[1]
    ax.bar(por_decil.index, por_decil.values, color='#2ca02c', alpha=0.85)
    ax.set_ylabel(f'{target_col} medio')
    ax.set_title(f'{target_col} medio por decil de {col}')
    ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return {
        'variable': col, 'target': target_col, 'n': len(sub),
        'pearson': pear_r, 'pearson_p': pear_p,
        'spearman': spear_r, 'spearman_p': spear_p,
        'direccion_fuerza': interpretar_fuerza(pear_r),
        'deciles': por_decil.to_dict(),  # datos detrás del panel de barras — se persisten, no solo se grafican
    }


total_graficos_q2_3 = sum(len(_variables_analisis_q2_3(tabla)) for tabla in tablas.values()) * len(TARGETS_Q2_3)
t_inicio_q2_3 = time.time()
contador_q2_3 = 0

resultados_q2_3 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            resultado = analizar_variable_vs_target(tabla, col, target_col, valor)
            contador_q2_3 += 1
            _progreso(contador_q2_3, total_graficos_q2_3, t_inicio_q2_3)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultados_q2_3.append(resultado)
            generados += 1
    print(f"{valor}: {generados} paneles generados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets).")

BTCUSD: 108 paneles generados, 18 omitidos (de 63 variables x 2 targets).


### 3.1. Caso aparte: `regimen_tendencia` (nominal, sin orden)

`bull`/`bear`/`sideways` no tiene un orden numérico natural — un
`factorize` asignaría códigos 0/1/2 según orden de aparición, no según
ninguna propiedad del régimen, y Pearson/Spearman sobre esos códigos no
sería interpretable como "dirección y fuerza de la asociación". En su lugar,
se comparan directamente las medias de `Close` y `retorno_log_1h` por
categoría — la misma pregunta de Q2-3 (dirección/fuerza), sin forzar un
orden que no existe.

In [8]:
for valor, tabla in tablas.items():
    medias_por_regimen = tabla.groupby(REGIMEN_NOMINAL, observed=True)[TARGETS_Q2_3].mean()
    print(f"{valor} — {TARGETS_Q2_3[0]}/{TARGETS_Q2_3[1]} medio por {REGIMEN_NOMINAL}:")
    print(medias_por_regimen)
    print()

BTCUSD — Close/retorno_log_1h medio por regimen_tendencia:
                          Close  retorno_log_1h
regimen_tendencia                              
bear               57239.909763       -0.000100
bull               58886.652673        0.000102
sideways           57708.884314        0.000010



### 3.2. Ranking inicial de Q2-3

Tabla de apoyo para escanear rápido los resultados anteriores, ordenada por
\|Pearson\| descendente dentro de cada `valor`/`target`. **No** es la síntesis
descriptiva completa (`Q2-11`, que combina además coseno, no linealidad,
régimen, estabilidad y redundancia) ni debe leerse como un ranking
predictivo — es solo qué tan fuerte se ha relacionado, en contemporáneo,
cada variable con el nivel (`Close`, referencial) o el retorno
(`retorno_log_1h`, principal) del precio.

In [9]:
ranking_q2_3 = (
    pd.DataFrame(resultados_q2_3)
    .drop(columns='deciles')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'target', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_3

,variable,target,n,pearson,pearson_p,spearman,spearman_p,direccion_fuerza,valor
0,drawdown_historico,Close,40283,0.866382,0.000000,0.888647,0.000000,positiva muy fuerte,BTCUSD
1,fundamental_edad,Close,40283,0.792445,0.000000,0.786920,0.000000,positiva muy fuerte,BTCUSD
2,x3_atr_14,Close,40283,0.674888,0.000000,0.768740,0.000000,positiva fuerte,BTCUSD
3,duracion_drawdown,Close,40283,-0.659814,0.000000,-0.748208,0.000000,negativa fuerte,BTCUSD
4,drawdown_365d,Close,40283,0.574979,0.000000,0.568211,0.000000,positiva fuerte,BTCUSD
...,...,...,...,...,...,...,...,...,...
103,ruido_ratio_movimiento_30d,retorno_log_1h,39563,-0.001303,0.795543,-0.005579,0.267124,negativa nula/despreciable,BTCUSD
104,ruido_atr_sin_direccion_30d,retorno_log_1h,39563,-0.001261,0.801940,-0.005918,0.239173,negativa nula/despreciable,BTCUSD
105,x3_bb_width,retorno_log_1h,40264,0.000847,0.864985,0.005107,0.305474,positiva nula/despreciable,BTCUSD
106,duracion_en_rango,retorno_log_1h,40282,0.000786,0.874612,-0.007001,0.159972,positiva nula/despreciable,BTCUSD


### 3.3. Guardar los datos detrás de los gráficos (no solo el render)

Los gráficos de esta sección no se commitean (ver nota de `[Q2-2]`/`[Q2-3]` en
`docs/done.md`) — pero los **datos** detrás de cada uno sí, para poder
analizarlos después sin tener que re-ejecutar el notebook completo: el
ranking de métricas (`ranking_q2_3`) y la tabla de medias por decil que
alimenta cada gráfico de barras (`deciles_q2_3`, formato largo:
`valor`/`variable`/`target`/`bin`/`media`). Quedan en `resources/x5_alt/exp_Q2/` —
mismo criterio que `[Q1-2]`/`[Q1-9]` — para que la celda de export a zip del
final del notebook (`Q2-5`/`Q2-EXEC`) los incluya junto con el resto.

In [10]:
deciles_q2_3 = pd.DataFrame([
    {'valor': r['valor'], 'variable': r['variable'], 'target': r['target'], 'bin': bin_label, 'media': media}
    for r in resultados_q2_3
    for bin_label, media in r['deciles'].items()
])

for valor in tablas:
    path_metricas = CARPETA_EXP / f'{valor}_q2_3_metricas.csv'
    path_deciles = CARPETA_EXP / f'{valor}_q2_3_deciles.csv'
    ranking_q2_3[ranking_q2_3['valor'] == valor].to_csv(path_metricas, index=False)
    deciles_q2_3[deciles_q2_3['valor'] == valor].to_csv(path_deciles, index=False)
    print(f"{valor}: métricas guardadas en {path_metricas.name}, deciles en {path_deciles.name}")

BTCUSD: métricas guardadas en BTCUSD_q2_3_metricas.csv, deciles en BTCUSD_q2_3_deciles.csv


## 4. Coseno vector-precio vs. vector-variable (Q2-4)

**Objetivo**: complementar Pearson/Spearman con una medida de alineación
direccional entre el vector de precios `Pv` y el de cada variable `Xv` en
el tiempo, `cos(Pv,Xv)` — la métrica nueva que introduce X5_alt_v2 respecto
a v1 (`x5_contexto_conversacion.md`, `X5_alternativo.md` §10.2).

**Pregunta**: ¿qué tan alineados están los vectores crudos de precio y
variable, sin centrarlos en su media?

**Decisión de diseño — mismos targets y variables que Q2-3**: en vez de
definir un criterio de selección nuevo, el coseno se calcula sobre
exactamente los mismos pares variable/target ya evaluados en la Sección 3
(`TARGETS_Q2_3` = Panel A `Close` referencial, Panel B `retorno_log_1h`
principal; `regimen_tendencia` sigue aparte, sin coseno, por la misma razón
de no tener orden numérico natural) — así se incorpora directamente al
ranking existente (`ranking_q2_4`, extensión de `ranking_q2_3`) en vez de
duplicar criterio.

**Diferencia conceptual con Pearson (por qué no es redundante)**: Pearson
centra ambas series en su propia media antes de medir covariación —
`corr(x,y) = cos(x - x̄, y - ȳ)`. El coseno de esta sección **no centra
nada**: mide el ángulo entre los vectores crudos. Esto produce dos
comportamientos muy distintos según el panel, verificados con los
resultados reales de BTCUSD (Sección 4.1):

- **Panel A (`Close`)**: no es cierto que el coseno sin centrar tienda
  simplemente a 1 — al mezclar el nivel compartido (ambos vectores son
  mayormente positivos) con la covariación real, el resultado puede
  **invertir el signo respecto a Pearson**. Ejemplos concretos de BTCUSD:
  `drawdown_historico` (`(Close - cummax(Close))/cummax(Close)`, ≤0, más
  cerca de 0 cuanto más cerca está el precio de su máximo histórico) tiene
  Pearson `+0.87` con `Close` (coherente: a lo largo del período el precio
  tiende a estar más alto justo cuando está más cerca de su máximo
  histórico) pero coseno `-0.51`; a la inversa, `ruido_cambios_signo_30d`
  tiene Pearson `-0.54` pero coseno `+0.87`. Esta métrica no es "una
  versión más permisiva de Pearson" sobre el nivel — es una cantidad
  distinta que mezcla baseline y covariación de un modo que no es
  comparable, razón suficiente para mantener Panel A como
  referencial/no concluyente también para el coseno, aunque por un motivo
  distinto al de Pearson (ahí el problema era la media compartida de
  ambas series por no estacionariedad; acá es que ninguna de las dos está
  centrada).
- **Panel B (`retorno_log_1h`)**: como el retorno ya es ~media cero, el
  coseno coincide en signo con Pearson y la magnitud difiere solo cuando
  la variable `Xv` tiene un baseline grande respecto a su propia
  dispersión — ej. `x3_rsi_14` (oscilador centrado ~50, no en 0): Pearson
  `0.338` vs. coseno `0.080`, fuertemente atenuado aunque la covariación
  real con el retorno sea la misma. Variables ya centradas en torno a 0
  (ej. `x3_macd_hist`, `x3_macd`) no sufren esta atenuación: Pearson y
  coseno salen prácticamente idénticos (`0.167` en ambos para
  `x3_macd_hist`).

**Interpretación**: el coseno se lee como complemento de alineación
direccional, no como reemplazo de Pearson/Spearman — se incorpora al
ranking (`ranking_q2_4`) para comparar lado a lado; la síntesis conjunta de
ambas métricas (junto a no linealidad, estabilidad, régimen y redundancia)
es responsabilidad de `Q2-11`, no de este bloque.

**Limitaciones**: mismas de Q2-3 — ~40.000 observaciones horarias
autocorrelacionadas (no independientes), variables con menos de 30
observaciones no nulas se omiten con aviso.

In [11]:
def coseno(x, y):
    """cos(Pv,Xv): a diferencia de Pearson, no centra en la media — ángulo entre los vectores crudos."""
    norm_x, norm_y = np.linalg.norm(x), np.linalg.norm(y)
    if norm_x == 0 or norm_y == 0:
        return np.nan
    return float(np.dot(x, y) / (norm_x * norm_y))


def analizar_coseno(df, col, target_col):
    """Mismo filtro de validez que `analizar_variable_vs_target` (Q2-3): >=30 obs., ambas series con dispersión."""
    sub = df[[col, target_col]].dropna()
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None
    return {'variable': col, 'target': target_col, 'n': len(sub), 'coseno': coseno(x, y)}


resultados_q2_4 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            resultado = analizar_coseno(tabla, col, target_col)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultados_q2_4.append(resultado)
            generados += 1
    print(f"{valor}: {generados} coseno calculados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets).")

BTCUSD: 108 coseno calculados, 18 omitidos (de 63 variables x 2 targets).


### 4.1. Ranking extendido: Pearson + Spearman + coseno (Q2-4)

Extiende `ranking_q2_3` (Sección 3.2) con la columna `coseno`, para leer
las tres métricas lado a lado sobre el mismo par variable/target — no
reemplaza ni recalcula Pearson/Spearman.

In [12]:
coseno_q2_4 = pd.DataFrame(resultados_q2_4)

ranking_q2_4 = (
    ranking_q2_3
    .merge(coseno_q2_4[['valor', 'variable', 'target', 'coseno']], on=['valor', 'variable', 'target'], how='left')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'target', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_4

,variable,target,n,pearson,pearson_p,spearman,spearman_p,direccion_fuerza,valor,coseno
0,drawdown_historico,Close,40283,0.866382,0.000000,0.888647,0.000000,positiva muy fuerte,BTCUSD,-0.508247
1,fundamental_edad,Close,40283,0.792445,0.000000,0.786920,0.000000,positiva muy fuerte,BTCUSD,0.950639
2,x3_atr_14,Close,40283,0.674888,0.000000,0.768740,0.000000,positiva fuerte,BTCUSD,0.914012
3,duracion_drawdown,Close,40283,-0.659814,0.000000,-0.748208,0.000000,negativa fuerte,BTCUSD,0.413424
4,drawdown_365d,Close,40283,0.574979,0.000000,0.568211,0.000000,positiva fuerte,BTCUSD,-0.503046
...,...,...,...,...,...,...,...,...,...,...
103,ruido_ratio_movimiento_30d,retorno_log_1h,39563,-0.001303,0.795543,-0.005579,0.267124,negativa nula/despreciable,BTCUSD,-0.001283
104,ruido_atr_sin_direccion_30d,retorno_log_1h,39563,-0.001261,0.801940,-0.005918,0.239173,negativa nula/despreciable,BTCUSD,-0.001241
105,x3_bb_width,retorno_log_1h,40264,0.000847,0.864985,0.005107,0.305474,positiva nula/despreciable,BTCUSD,0.000865
106,duracion_en_rango,retorno_log_1h,40282,0.000786,0.874612,-0.007001,0.159972,positiva nula/despreciable,BTCUSD,0.000992


### 4.2. Guardar el ranking con coseno

Propio de `[Q2-4]` (no sobrescribe `{valor}_q2_3_metricas.csv`) — mismo
criterio de un CSV por ítem que `[Q2-3]`/`[Q1-2]`/`[Q1-9]`, para que la
celda de export a zip del final del notebook (`Q2-5`/`Q2-EXEC`) lo
incluya.

In [13]:
for valor in tablas:
    path_q2_4 = CARPETA_EXP / f'{valor}_q2_4_ranking_coseno.csv'
    ranking_q2_4[ranking_q2_4['valor'] == valor].to_csv(path_q2_4, index=False)
    print(f"{valor}: ranking con coseno guardado en {path_q2_4.name}")

BTCUSD: ranking con coseno guardado en BTCUSD_q2_4_ranking_coseno.csv


## 5. No linealidades y umbrales (Q2-7)

**Objetivo**: ampliar el análisis variable↔target de la Sección 3 (`Q2-3`)
más allá de la forma lineal — detectar curvatura, comportamiento distinto
en los extremos de la variable, posibles umbrales/saturaciones y
asimetría, sin asumir que Pearson/regresión lineal capturan toda la
relación (`x5_contexto_conversacion.md` §15,
`solicitud_claude_code_X5_P2_notebook_v2.md` §4.9).

**Pregunta**: ¿la relación entre cada variable y el target (nivel o
retorno) es realmente lineal en todo su rango, o cambia de forma en algún
tramo — por ejemplo, plana en valores normales pero distinta sobre el
percentil 90, o con un efecto que se satura en los extremos en vez de
seguir creciendo?

**Decisión de diseño — mismos pares variable/target que Q2-3/Q2-4**: igual
que en la Sección 4, se reutilizan exactamente `TARGETS_Q2_3` y
`_variables_analisis_q2_3` (Panel A `Close` referencial, Panel B
`retorno_log_1h` principal; `regimen_tendencia` queda fuera — ya se trató
aparte en 3.1 y no tiene un rango continuo sobre el que evaluar
curvatura/colas) — así el resultado se incorpora al mismo ranking que
viene extendiéndose (`ranking_q2_7`, extensión de `ranking_q2_4`) en vez
de duplicar criterio de selección.

**Metodología** (por variable, por panel):
- **Curva por cuantil fino** (20 bins, más fino que los 10 de Q2-3):
  media del target por bin de cuantil de la variable, usando el valor
  medio real de la variable en cada bin (no solo una etiqueta `b1..b20`)
  — es la "curva suavizada" que revela la forma real sin asumir
  linealidad.
- **Curvatura — cuadrático vs. lineal**: ajuste `polyfit` grado 1 y
  grado 2 sobre los mismos puntos de Q2-3; `delta_r2 = R²_cuad - R²_lin`
  mide cuánta varianza extra explica la curvatura; el signo del
  coeficiente cuadrático indica forma U (convexa) o U invertida (cóncava)
  cuando `delta_r2` supera `UMBRAL_DELTA_R2=0.01`.
- **Comportamiento en extremos / umbral**: se definen tres tramos de la
  variable según sus percentiles 10/90 (cola baja, cuerpo, cola alta); si
  la media del target en una cola se aparta de la media del cuerpo en más
  de `UMBRAL_DESVIOS_COLA=0.5` desviaciones estándar (calculadas sobre el
  cuerpo), se marca esa cola como candidata a umbral — el caso del
  ejemplo del plan ("sin relación en valores normales pero distinta
  sobre el percentil 90").
- **Saturación**: se compara la pendiente lineal local de cada cola
  contra la pendiente global; si la pendiente de una cola tiene el mismo
  signo que la global pero su magnitud cae por debajo de
  `UMBRAL_RATIO_SATURACION=0.3` de la pendiente global, se marca esa cola
  como saturada (el efecto sigue en la misma dirección pero se aplana).
- **Asimetría**: `skew` (asimetría de la distribución) de la propia
  variable, más una comparación directa del Pearson calculado solo bajo
  la mediana vs. solo sobre la mediana — si ambos difieren marcadamente
  (incluyendo cambio de signo), la relación no es simétrica respecto al
  valor típico de la variable.

**No se duplica el scatter de la Sección 3** — el foco nuevo de esta
sección es la curva por cuantil fino (expone la forma no lineal mejor
que el decil grueso de Q2-3) y las métricas de curvatura/cola/asimetría
recién descritas. Mínimo `10` observaciones por cola para calcular su
métrica (`MIN_OBS_COLA`); si no se alcanza, queda en `NaN`.

**Variables analizadas**: las mismas de `Q2-3`/`Q2-4`
(`_variables_analisis_q2_3` × `TARGETS_Q2_3`), con el mismo filtro de
`n >= 30` observaciones no nulas y dispersión no nula.

**Interpretación**: una variable con `delta_r2` alto, con una cola
marcada como umbral, o con asimetría fuerte entre mitades, es candidata a
jugar un rol de "régimen"/condición (activar o modular un parámetro de
X5 solo bajo cierto percentil) en vez de una relación lineal continua —
alimenta la síntesis (`Q2-11`) y las hipótesis de parámetros con
comportamiento de umbral (`K`, `N_EXP`, ver `X5_alternativo.md` §10.5).
Esto es descriptivo/contemporáneo, igual que Q2-3/Q2-4 — no implica
capacidad predictiva sobre el futuro.

**Limitaciones**: mismas de Q2-3/Q2-4 (observaciones horarias
autocorrelacionadas, p-values no literales, variables con menos de 30
observaciones no nulas se omiten); el ajuste cuadrático es descriptivo,
no un test formal de forma funcional; con colas muy angostas las medias
de cola pueden salir de muy pocas observaciones, de ahí el mínimo de
`MIN_OBS_COLA=10`.

In [14]:
N_BINS_FINO = 20  # más fino que N_BINS=10 (Q2-3) — resuelve mejor la forma de la curva
P_COLA = 0.10  # percentil que define cola baja (<=p10) / cola alta (>=p90)
UMBRAL_DELTA_R2 = 0.01  # mejora de R² del cuadrático sobre el lineal a partir de la cual se considera curvatura relevante
UMBRAL_DESVIOS_COLA = 0.5  # desvíos estándar (del cuerpo) que debe apartarse la media de una cola para marcarla como candidata a umbral
UMBRAL_RATIO_SATURACION = 0.3  # pendiente local de la cola por debajo de esta fracción de la pendiente global → saturación
MIN_OBS_COLA = 10  # mínimo de observaciones en una cola para calcular su métrica; si no se alcanza, queda en NaN


def analizar_no_linealidad(df, col, target_col, valor, n_bins=N_BINS_FINO, p_cola=P_COLA):
    """Q2-7: curvatura (cuadrático vs. lineal), comportamiento en colas (umbral/saturación) y asimetría."""
    sub = df[[col, target_col]].dropna()
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None

    coef_lin = np.polyfit(x, y, 1)
    coef_cuad = np.polyfit(x, y, 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2_lin = 1 - np.sum((y - np.polyval(coef_lin, x)) ** 2) / ss_tot
    r2_cuad = 1 - np.sum((y - np.polyval(coef_cuad, x)) ** 2) / ss_tot
    delta_r2 = r2_cuad - r2_lin
    if delta_r2 < UMBRAL_DELTA_R2:
        forma = 'sin curvatura relevante'
    elif coef_cuad[0] > 0:
        forma = 'U (convexa)'
    else:
        forma = 'U invertida (cóncava)'

    try:
        bins = pd.qcut(sub[col], n_bins, duplicates='drop')
    except ValueError:
        bins = pd.qcut(sub[col].rank(method='first'), n_bins)
    curva = sub.groupby(bins, observed=True).agg(x_medio=(col, 'mean'), y_medio=(target_col, 'mean')).reset_index(drop=True)
    curva.insert(0, 'bin', [f"b{i + 1}" for i in range(len(curva))])

    p_bajo, p_alto = np.quantile(x, p_cola), np.quantile(x, 1 - p_cola)
    mask_baja, mask_alta = x <= p_bajo, x >= p_alto
    mask_cuerpo = ~(mask_baja | mask_alta)
    media_cuerpo, std_cuerpo = y[mask_cuerpo].mean(), y[mask_cuerpo].std()
    pendiente_global = coef_lin[0]

    media_cola_baja = y[mask_baja].mean() if mask_baja.sum() >= MIN_OBS_COLA else np.nan
    media_cola_alta = y[mask_alta].mean() if mask_alta.sum() >= MIN_OBS_COLA else np.nan
    umbral_cola_baja = bool(std_cuerpo and np.isfinite(media_cola_baja) and abs(media_cola_baja - media_cuerpo) > UMBRAL_DESVIOS_COLA * std_cuerpo)
    umbral_cola_alta = bool(std_cuerpo and np.isfinite(media_cola_alta) and abs(media_cola_alta - media_cuerpo) > UMBRAL_DESVIOS_COLA * std_cuerpo)

    pendiente_cola_baja = np.polyfit(x[mask_baja], y[mask_baja], 1)[0] if mask_baja.sum() >= MIN_OBS_COLA and x[mask_baja].std() > 0 else np.nan
    pendiente_cola_alta = np.polyfit(x[mask_alta], y[mask_alta], 1)[0] if mask_alta.sum() >= MIN_OBS_COLA and x[mask_alta].std() > 0 else np.nan
    saturacion_cola_baja = bool(pendiente_global and np.isfinite(pendiente_cola_baja) and np.sign(pendiente_cola_baja) == np.sign(pendiente_global) and abs(pendiente_cola_baja / pendiente_global) < UMBRAL_RATIO_SATURACION)
    saturacion_cola_alta = bool(pendiente_global and np.isfinite(pendiente_cola_alta) and np.sign(pendiente_cola_alta) == np.sign(pendiente_global) and abs(pendiente_cola_alta / pendiente_global) < UMBRAL_RATIO_SATURACION)

    mediana_x = np.median(x)
    mask_mitad_baja, mask_mitad_alta = x < mediana_x, x >= mediana_x
    pearson_mitad_baja = scipy_stats.pearsonr(x[mask_mitad_baja], y[mask_mitad_baja])[0] if mask_mitad_baja.sum() >= MIN_OBS_COLA else np.nan
    pearson_mitad_alta = scipy_stats.pearsonr(x[mask_mitad_alta], y[mask_mitad_alta])[0] if mask_mitad_alta.sum() >= MIN_OBS_COLA else np.nan

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"{valor} — {col} vs. {target_col} — {forma} (ΔR²={delta_r2:.3f})", fontsize=10)

    ax = axes[0]
    ax.plot(curva['x_medio'], curva['y_medio'], marker='o', markersize=3, linewidth=1, color='#2ca02c', label=f'media por bin ({n_bins} cuantiles)')
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, np.polyval(coef_lin, x_line), color='#1f77b4', linewidth=1, linestyle='--', label='ajuste lineal')
    ax.plot(x_line, np.polyval(coef_cuad, x_line), color='#d62728', linewidth=1.3, label='ajuste cuadrático')
    ax.axvline(p_bajo, color='gray', linewidth=0.8, linestyle=':')
    ax.axvline(p_alto, color='gray', linewidth=0.8, linestyle=':')
    ax.set_xlabel(col)
    ax.set_ylabel(target_col)
    ax.set_title('Curva por cuantil fino + ajustes')
    ax.legend(fontsize=7)

    ax = axes[1]
    tramos = ['cola baja', 'cuerpo', 'cola alta']
    ax.bar(tramos, [media_cola_baja, media_cuerpo, media_cola_alta], color=['#9467bd', '#7f7f7f', '#ff7f0e'], alpha=0.85)
    ax.set_ylabel(f'{target_col} medio')
    ax.set_title(f'Media por tramo (p{int(p_cola * 100)}/p{int((1 - p_cola) * 100)})')

    plt.tight_layout()
    plt.show()

    return {
        'variable': col, 'target': target_col, 'n': len(sub),
        'r2_lin': r2_lin, 'r2_cuad': r2_cuad, 'delta_r2': delta_r2, 'forma': forma,
        'media_cuerpo': media_cuerpo, 'media_cola_baja': media_cola_baja, 'media_cola_alta': media_cola_alta,
        'umbral_cola_baja': umbral_cola_baja, 'umbral_cola_alta': umbral_cola_alta,
        'saturacion_cola_baja': saturacion_cola_baja, 'saturacion_cola_alta': saturacion_cola_alta,
        'asimetria_variable': float(scipy_stats.skew(x)),
        'pearson_mitad_baja': pearson_mitad_baja, 'pearson_mitad_alta': pearson_mitad_alta,
        'curva': curva.to_dict('records'),
    }

In [15]:
total_graficos_q2_7 = sum(len(_variables_analisis_q2_3(tabla)) for tabla in tablas.values()) * len(TARGETS_Q2_3)
t_inicio_q2_7 = time.time()
contador_q2_7 = 0

resultados_q2_7 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            resultado = analizar_no_linealidad(tabla, col, target_col, valor)
            contador_q2_7 += 1
            _progreso(contador_q2_7, total_graficos_q2_7, t_inicio_q2_7)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultados_q2_7.append(resultado)
            generados += 1
    print(f"{valor}: {generados} paneles de no linealidad generados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets).")

/var/folders/hg/lrpyrdsj0_xgmj2pfk8cds740000gn/T/ipykernel_69441/1249725880.py:41: RuntimeWarning: Mean of empty slice.
  media_cuerpo, std_cuerpo = y[mask_cuerpo].mean(), y[mask_cuerpo].std()
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/var/folders/hg/lrpyrdsj0_xgmj2pfk8cds740000gn/T/ipyker

/var/folders/hg/lrpyrdsj0_xgmj2pfk8cds740000gn/T/ipykernel_69441/1249725880.py:41: RuntimeWarning: Mean of empty slice.
  media_cuerpo, std_cuerpo = y[mask_cuerpo].mean(), y[mask_cuerpo].std()
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:180: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/var/folders/hg/lrpyrdsj0_xgmj2pfk8cds740000gn/T/ipyker

BTCUSD: 108 paneles de no linealidad generados, 18 omitidos (de 63 variables x 2 targets).


### 5.1. Ranking extendido: + no linealidad, umbrales, saturación y asimetría (Q2-7)

Extiende `ranking_q2_4` (Secciones 3.2/4.1) con las columnas de esta
sección (`forma`, `delta_r2`, `umbral_cola_baja/alta`,
`saturacion_cola_baja/alta`, `asimetria_variable`,
`pearson_mitad_baja/alta`) — no reemplaza ni recalcula Pearson/Spearman/
coseno.

In [16]:
columnas_q2_7 = [
    'valor', 'variable', 'target', 'r2_lin', 'r2_cuad', 'delta_r2', 'forma',
    'media_cuerpo', 'media_cola_baja', 'media_cola_alta',
    'umbral_cola_baja', 'umbral_cola_alta', 'saturacion_cola_baja', 'saturacion_cola_alta',
    'asimetria_variable', 'pearson_mitad_baja', 'pearson_mitad_alta',
]
no_linealidad_q2_7 = pd.DataFrame(resultados_q2_7)[columnas_q2_7]

ranking_q2_7 = (
    ranking_q2_4
    .merge(no_linealidad_q2_7, on=['valor', 'variable', 'target'], how='left')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'target', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_7

,variable,target,n,pearson,pearson_p,spearman,spearman_p,direccion_fuerza,valor,coseno,...,media_cuerpo,media_cola_baja,media_cola_alta,umbral_cola_baja,umbral_cola_alta,saturacion_cola_baja,saturacion_cola_alta,asimetria_variable,pearson_mitad_baja,pearson_mitad_alta
0,drawdown_historico,Close,40283,0.866382,0.000000,0.888647,0.000000,positiva muy fuerte,BTCUSD,-0.508247,...,5.794897e+04,18702.085567,97139.986905,True,True,False,False,-0.121467,0.800873,0.496154
1,fundamental_edad,Close,40283,0.792445,0.000000,0.786920,0.000000,positiva muy fuerte,BTCUSD,0.950639,...,5.691401e+04,46858.212038,77261.790759,False,True,False,False,0.000000,-0.242296,0.440221
2,x3_atr_14,Close,40283,0.674888,0.000000,0.768740,0.000000,positiva fuerte,BTCUSD,0.914012,...,5.857900e+04,22663.585719,88139.366451,True,True,False,False,1.128648,0.604402,0.244979
3,duracion_drawdown,Close,40283,-0.659814,0.000000,-0.748208,0.000000,negativa fuerte,BTCUSD,0.413424,...,5.575158e+04,93549.622074,39811.456821,True,True,False,False,0.915252,-0.519812,-0.227145
4,drawdown_365d,Close,40283,0.574979,0.000000,0.568211,0.000000,positiva fuerte,BTCUSD,-0.503046,...,6.164658e+04,19312.734686,66954.871167,True,False,False,False,-0.712308,0.736105,0.219209
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,ruido_ratio_movimiento_30d,retorno_log_1h,39563,-0.001303,0.795543,-0.005579,0.267124,negativa nula/despreciable,BTCUSD,-0.001283,...,-6.715927e-06,0.000057,0.000053,False,False,False,False,166.110064,-0.008176,-0.002140
104,ruido_atr_sin_direccion_30d,retorno_log_1h,39563,-0.001261,0.801940,-0.005918,0.239173,negativa nula/despreciable,BTCUSD,-0.001241,...,2.578380e-07,0.000023,0.000030,False,False,False,False,167.091940,-0.005957,-0.002046
105,x3_bb_width,retorno_log_1h,40264,0.000847,0.864985,0.005107,0.305474,positiva nula/despreciable,BTCUSD,0.000865,...,-6.216663e-06,0.000029,0.000045,False,False,False,False,2.221781,-0.009670,-0.002311
106,duracion_en_rango,retorno_log_1h,40282,0.000786,0.874612,-0.007001,0.159972,positiva nula/despreciable,BTCUSD,0.000992,...,4.440020e-06,-0.000004,0.000006,False,False,False,False,1.726725,0.003172,0.001361


### 5.2. Guardar el ranking extendido y las curvas por cuantil fino

Mismo criterio de un CSV por ítem que las secciones anteriores. La curva
por cuantil fino (`curva`, formato largo: `valor`/`variable`/`target`/
`bin`/`x_medio`/`y_medio`) se guarda aparte — es el dato detrás del panel
izquierdo de cada gráfico de esta sección, análogo a
`{valor}_q2_3_deciles.csv`.

In [17]:
curvas_q2_7 = pd.DataFrame([
    {'valor': r['valor'], 'variable': r['variable'], 'target': r['target'], **fila}
    for r in resultados_q2_7
    for fila in r['curva']
])

for valor in tablas:
    path_ranking = CARPETA_EXP / f'{valor}_q2_7_ranking.csv'
    path_curvas = CARPETA_EXP / f'{valor}_q2_7_curvas.csv'
    ranking_q2_7[ranking_q2_7['valor'] == valor].to_csv(path_ranking, index=False)
    curvas_q2_7[curvas_q2_7['valor'] == valor].to_csv(path_curvas, index=False)
    print(f"{valor}: ranking extendido guardado en {path_ranking.name}, curvas en {path_curvas.name}")

BTCUSD: ranking extendido guardado en BTCUSD_q2_7_ranking.csv, curvas en BTCUSD_q2_7_curvas.csv


## 6. Variable vs. comportamiento histórico reciente del precio (Q2-8)

**Objetivo**: caracterizar qué venía ocurriendo con el precio en el pasado
reciente — antes de `t`, no en `t` — cuando cada variable de estado toma
determinados valores. A diferencia de la Sección 3 (`Q2-3`, contemporáneo:
`X_t` vs. `retorno_t` en la misma vela), aquí el target es el retorno que el
precio ya acumuló en una ventana hacia atrás:
`retorno_pasado_{h} = log(Close_t / Close_{t-h})`.

**Pregunta**: cuando la variable toma determinado valor, ¿qué tipo de
movimiento venía arrastrando el precio en las horas/días previos?

**Horizontes**: `6h, 24h, 3d (72h), 7d (168h)` — los mismos de
`x5_contexto_conversacion.md` §15 ("retornos acumulados a 6h, 24h, 3d y 7d"),
expresados en velas H1 (granularidad de la tabla de Q1).

**Decisión de diseño — mismas variables que Q2-3, nuevo target por
horizonte**: se reutiliza `_variables_analisis_q2_3` (excluye columnas de
precio crudo, `regimen_tendencia` y los targets derivados ya calculados en el
notebook) y la misma función `analizar_variable_vs_target` de la Sección 3
(Pearson, Spearman, scatter + regresión, target medio por decil) — ya
genérica en `target_col`, no se duplica lógica: solo se la llama con cada
`retorno_pasado_{h}` en vez de `Close`/`retorno_log_1h`.

**Interpretación**: este bloque sigue siendo descriptivo/histórico — no dice
nada sobre el retorno *futuro* (eso es Parte II, fuera del alcance de Q2).
Una variable fuertemente asociada al retorno pasado de 7 días puede, por
ejemplo, estar simplemente *midiendo* ese retorno (caso de variables como
`x3_roc_20` o los `drawdown_*`, construidas a partir del propio precio) — la
Sección 10 (`Q2-12`, redundancia) es la que distingue eso de una relación
informativa nueva.

**Limitaciones**: mismas de la Sección 3 (autocorrelación temporal → p-values
no confiables bajo independencia; correlación no implica causalidad); además,
a mayor horizonte `h` las ventanas se superponen más entre sí, por lo que hay
menos observaciones realmente independientes que velas en el período total.
No se genera un ranking propio por horizonte agregando todos los `h` en una
sola tabla ordenable — se deja uno por horizonte, para no mezclar escalas
temporales distintas bajo un solo `abs(pearson)`.

In [18]:
HORIZONTES_Q2_8 = {'6h': 6, '24h': 24, '3d': 72, '7d': 168}  # en velas H1

for tabla in tablas.values():
    for etiqueta, h in HORIZONTES_Q2_8.items():
        tabla[f'retorno_pasado_{etiqueta}'] = np.log(tabla['Close'] / tabla['Close'].shift(h))

TARGETS_DERIVADOS.update(f'retorno_pasado_{etiqueta}' for etiqueta in HORIZONTES_Q2_8)

print(f"Horizontes evaluados: {list(HORIZONTES_Q2_8.keys())}")

Horizontes evaluados: ['6h', '24h', '3d', '7d']


In [19]:
total_graficos_q2_8 = sum(len(_variables_analisis_q2_3(tabla)) for tabla in tablas.values()) * len(HORIZONTES_Q2_8)
t_inicio_q2_8 = time.time()
contador_q2_8 = 0

resultados_q2_8 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for etiqueta, h in HORIZONTES_Q2_8.items():
        target_col = f'retorno_pasado_{etiqueta}'
        for col in variables_analisis:
            resultado = analizar_variable_vs_target(tabla, col, target_col, valor)
            contador_q2_8 += 1
            _progreso(contador_q2_8, total_graficos_q2_8, t_inicio_q2_8)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultado['horizonte_h'] = h
            resultados_q2_8.append(resultado)
            generados += 1
    print(f"{valor}: {generados} paneles generados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(HORIZONTES_Q2_8)} horizontes).")

BTCUSD: 216 paneles generados, 36 omitidos (de 63 variables x 4 horizontes).


### 6.1. Ranking de Q2-8 por horizonte

Un ranking por `valor`/horizonte (no uno global mezclando horizontes), igual
criterio de `abs(pearson)` descendente que la Sección 3.2.

In [20]:
ranking_q2_8 = (
    pd.DataFrame(resultados_q2_8)
    .drop(columns='deciles')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'horizonte_h', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_8

,variable,target,n,pearson,pearson_p,spearman,spearman_p,direccion_fuerza,valor,horizonte_h
0,x3_ema_dist_12,retorno_pasado_6h,40277,0.893441,0.000000,0.832654,0.000000e+00,positiva muy fuerte,BTCUSD,6
1,distancia_ema_12,retorno_pasado_6h,40277,0.814995,0.000000,0.828395,0.000000e+00,positiva muy fuerte,BTCUSD,6
2,x3_sma_dist_20,retorno_pasado_6h,40264,0.813027,0.000000,0.731570,0.000000e+00,positiva muy fuerte,BTCUSD,6
3,x3_roc_10,retorno_pasado_6h,40273,0.772308,0.000000,0.702759,0.000000e+00,positiva muy fuerte,BTCUSD,6
4,x3_ema_dist_26,retorno_pasado_6h,40277,0.768317,0.000000,0.674147,0.000000e+00,positiva muy fuerte,BTCUSD,6
...,...,...,...,...,...,...,...,...,...,...
211,ruido_mecha_30d,retorno_pasado_7d,40115,-0.012883,0.009872,-0.039096,4.757085e-15,negativa nula/despreciable,BTCUSD,168
212,x3_macd_hist,retorno_pasado_7d,40115,0.012176,0.014739,-0.026651,9.366962e-08,positiva nula/despreciable,BTCUSD,168
213,vol_alcista_30d,retorno_pasado_7d,40115,-0.007945,0.111569,0.029857,2.216321e-09,negativa nula/despreciable,BTCUSD,168
214,ruido_ratio_movimiento_30d,retorno_pasado_7d,39563,0.003282,0.513932,-0.079816,6.266751e-57,positiva nula/despreciable,BTCUSD,168


### 6.2. Guardar el ranking y los deciles por horizonte

Mismo criterio que `[Q2-3]`: el ranking de métricas y la tabla de medias por
decil (formato largo) se persisten en `resources/x5_alt/exp_Q2/`, para que la celda
de export a zip del final del notebook (`Q2-5`/`Q2-EXEC`) las incluya.

In [21]:
deciles_q2_8 = pd.DataFrame([
    {'valor': r['valor'], 'variable': r['variable'], 'target': r['target'], 'horizonte_h': r['horizonte_h'],
     'bin': bin_label, 'media': media}
    for r in resultados_q2_8
    for bin_label, media in r['deciles'].items()
])

for valor in tablas:
    path_ranking = CARPETA_EXP / f'{valor}_q2_8_ranking.csv'
    path_deciles = CARPETA_EXP / f'{valor}_q2_8_deciles.csv'
    ranking_q2_8[ranking_q2_8['valor'] == valor].to_csv(path_ranking, index=False)
    deciles_q2_8[deciles_q2_8['valor'] == valor].to_csv(path_deciles, index=False)
    print(f"{valor}: ranking de retorno pasado guardado en {path_ranking.name}, deciles en {path_deciles.name}")

BTCUSD: ranking de retorno pasado guardado en BTCUSD_q2_8_ranking.csv, deciles en BTCUSD_q2_8_deciles.csv


## 7. Ceteris paribus histórico — regresión multivariable controlando por otras variables (Q2-6)

**Objetivo**: estimar el efecto de cada variable de estado sobre el target
*manteniendo fijas las demás*. La Sección 3 (`Q2-3`) midió cada variable
contra el target una a la vez, pero varias variables de la tabla de Q1
comparten la misma información (distintas distancias a SMA, drawdown a
distinta ventana, etc.), así que un Pearson individual alto puede ser la
misma señal "contada dos veces" por dos columnas distintas.

**Pregunta**: si comparo dos instantes con el MISMO valor en todas las demás
variables y que solo difieran en una, ¿cambia el precio/retorno?

**Decisión de diseño — mismas variables y targets que Q2-3**: mismo
`_variables_analisis_q2_3` (excluye `regimen_tendencia`, sin orden numérico, y
los targets derivados) como regresores, y los mismos `TARGETS_Q2_3` (Panel A
`Close` referencial, Panel B `retorno_log_1h` principal) — no se introduce un
criterio de selección nuevo.

**Metodología**: regresión lineal múltiple con todas las variables
estandarizadas (z-score) — mismo enfoque manual con `numpy`/`scipy` (mínimos
cuadrados + errores estándar clásicos de OLS, sin `statsmodels`) de
`Otros/scripts/X5_analisis_exploratorio.ipynb` (`regresion_ceteris_paribus`),
adaptado acá a los targets/variables de Q2 — no se importa desde `Otros/`
para no acoplar X5_alt_v2 a la implementación de X5 original. El coeficiente
de cada variable es el cambio esperado en el target (en desvíos estándar)
cuando esa variable sube 1 desvío estándar, con las demás fijas.

**Multicolinealidad**: Factor de Inflación de Varianza (VIF) de cada
variable, calculado como la diagonal de la inversa de la matriz de
correlación de las variables estandarizadas (equivalente a `1/(1-R²)` de
regresarla contra todas las demás, sin correr una regresión por variable).
Convención: VIF > 10 señala colinealidad alta — el coeficiente individual de
esa variable es poco confiable, sin que eso implique eliminarla.

**Sensibilidad a especificaciones**: se repite la regresión excluyendo las
variables con VIF > 10 (especificación reducida) y se compara el coeficiente
de las que sobreviven en ambas — un cambio de signo indica que el resultado
depende de qué colineales se incluyen, no un efecto robusto.

**Estabilidad**: en este bloque se mide como la consistencia de signo del
coeficiente entre la especificación completa y la reducida — la estabilidad
*temporal* (por período) es responsabilidad de la Sección 8 (`Q2-9`), no de
este bloque.

**Interpretación**: se cruza el coeficiente ceteris paribus (especificación
completa) con el Pearson crudo de la Sección 3 (`ranking_q2_3`) para
identificar qué variables mantienen signo/fuerza al controlar por las demás y
cuáles ven su relación debilitada o invertida.

**Limitaciones**: las de la Sección 3 (autocorrelación temporal, no
causalidad); un modelo lineal aditivo no captura interacciones ni no
linealidades (eso es la Sección 5, `Q2-7`); con variables 100% nulas en este
snapshot de Mac (`x2_score` y derivadas, `fundamental_cambio` y derivadas —
ver limitación ya documentada en `[Q1-10]`/`[Q2-2]`) se excluyen de la
regresión antes de calcular `dropna()`, para que no arrastren a cero las
filas disponibles de todas las demás variables.

In [22]:
def calcular_vif(df, cols):
    """VIF de cada columna vía la diagonal de la inversa de su matriz de correlación
    (equivalente a 1/(1-R²) de regresarla contra las demás, sin correr N regresiones).
    Excluye antes las columnas 100% nulas o sin varianza — una sola columna constante
    deja NaN toda su fila/columna en la matriz de correlación y rompe la inversión para
    TODAS las demás variables, no solo para ella."""
    cols_con_dato = [c for c in cols if df[c].notna().any()]
    sub = df[cols_con_dato].dropna()
    sd = sub.std(numeric_only=True)
    cols_validos = sd[sd > 1e-12].index.tolist()
    descartados = sorted(set(cols_con_dato) - set(cols_validos))
    if descartados:
        print(f"  (!) Variables sin varianza excluidas del VIF: {descartados}")
    corr = sub[cols_validos].corr(numeric_only=True).to_numpy()
    vif = np.diag(np.linalg.pinv(corr))
    return pd.Series(vif, index=cols_validos, name='VIF').sort_values(ascending=False)


def regresion_ceteris_paribus(df, feature_cols, target_col, min_filas=100):
    """Regresión lineal múltiple estandarizada (z-score) — mínimos cuadrados + errores
    estándar de OLS calculados a mano (ver Otros/scripts/X5_analisis_exploratorio.ipynb,
    adaptado acá a los targets/variables de Q2 en vez de copiarlo literalmente)."""
    cols = [c for c in feature_cols if c in df.columns]
    cols_sin_dato = [c for c in cols if df[c].notna().sum() == 0]
    if cols_sin_dato:
        print(f"  (!) Variables 100% nulas excluidas de la regresión ({target_col}): {cols_sin_dato}")
    cols = [c for c in cols if c not in cols_sin_dato]

    sub = df[cols + [target_col]].dropna()
    n = len(sub)
    if n < min_filas:
        print(f"  (!) Solo {n} filas sin NaN (mínimo {min_filas}). Se omite la regresión para {target_col}.")
        return None

    X = sub[cols].to_numpy(dtype=float)
    y = sub[target_col].to_numpy(dtype=float)

    sd_x = X.std(axis=0, ddof=0)
    cols_validas = sd_x > 1e-12
    if not cols_validas.all():
        descartadas = [c for c, ok in zip(cols, cols_validas) if not ok]
        print(f"  (!) Columnas sin varianza (constantes) descartadas: {descartadas}")
    cols = [c for c, ok in zip(cols, cols_validas) if ok]
    X = X[:, cols_validas]

    mu_x, sd_x = X.mean(axis=0), X.std(axis=0, ddof=0)
    Xz = (X - mu_x) / sd_x
    sd_y = y.std(ddof=0)
    sd_y = sd_y if sd_y > 1e-12 else 1.0
    yz = (y - y.mean()) / sd_y

    X_dis = np.column_stack([np.ones(n), Xz])
    beta, *_ = np.linalg.lstsq(X_dis, yz, rcond=None)
    resid = yz - X_dis @ beta

    dof = n - X_dis.shape[1]
    if dof <= 0:
        print("  (!) No quedan grados de libertad. Se omite la regresión.")
        return None
    sigma2 = (resid @ resid) / dof
    XtX_inv = np.linalg.pinv(X_dis.T @ X_dis)
    se = np.sqrt(np.clip(np.diag(sigma2 * XtX_inv), 0, None))
    t_val = np.divide(beta, se, out=np.zeros_like(beta), where=se > 0)
    p_val = 2 * scipy_stats.t.sf(np.abs(t_val), df=dof)

    ss_tot = ((yz - yz.mean()) ** 2).sum()
    r2 = 1 - (resid @ resid) / ss_tot if ss_tot > 1e-12 else np.nan

    tabla_out = pd.DataFrame({
        'feature': ['(intercepto)'] + cols,
        'coef_ceteris_paribus': beta,
        'error_estandar': se,
        't': t_val,
        'p_valor': p_val,
    })
    tabla_out = tabla_out[tabla_out['feature'] != '(intercepto)'].reset_index(drop=True)
    tabla_out['significativo_5pct'] = tabla_out['p_valor'] < 0.05
    tabla_out.attrs['r2'], tabla_out.attrs['n'] = r2, n
    return tabla_out

In [23]:
MIN_FILAS_CETERIS = 100
UMBRAL_VIF_ALTO = 10

resultados_q2_6, vif_q2_6_partes = [], []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    vif_valor = calcular_vif(tabla, variables_analisis)
    vif_q2_6_partes.append(vif_valor.rename_axis('variable').reset_index().assign(valor=valor))
    vars_vif_alto = vif_valor[vif_valor > UMBRAL_VIF_ALTO].index.tolist()
    print(f"{valor}: {len(vars_vif_alto)}/{len(vif_valor)} variables con VIF > {UMBRAL_VIF_ALTO}.")

    for target_col in TARGETS_Q2_3:
        tabla_completa = regresion_ceteris_paribus(tabla, variables_analisis, target_col, MIN_FILAS_CETERIS)
        if tabla_completa is None:
            continue
        variables_reducidas = [c for c in variables_analisis if c not in vars_vif_alto]
        tabla_reducida = regresion_ceteris_paribus(tabla, variables_reducidas, target_col, MIN_FILAS_CETERIS)

        tabla_merge = tabla_completa.copy()
        if tabla_reducida is not None:
            tabla_merge = tabla_merge.merge(
                tabla_reducida[['feature', 'coef_ceteris_paribus']].rename(columns={'coef_ceteris_paribus': 'coef_reducida'}),
                on='feature', how='left')
        else:
            tabla_merge['coef_reducida'] = np.nan
        tabla_merge['cambia_signo_al_reducir'] = (
            (np.sign(tabla_merge['coef_ceteris_paribus']) != np.sign(tabla_merge['coef_reducida']))
            & tabla_merge['coef_reducida'].notna()
        )
        tabla_merge['valor'], tabla_merge['target'] = valor, target_col
        tabla_merge['vif'] = tabla_merge['feature'].map(vif_valor)
        resultados_q2_6.append(tabla_merge)
        n_cambia = int(tabla_merge['cambia_signo_al_reducir'].sum())
        print(f"{valor}/{target_col}: R²={tabla_completa.attrs['r2']:.3f} (n={tabla_completa.attrs['n']}), "
              f"{n_cambia} variable(s) cambian de signo al pasar a la especificación reducida.")

ceteris_q2_6 = pd.concat(resultados_q2_6, ignore_index=True) if resultados_q2_6 else pd.DataFrame()
vif_q2_6 = pd.concat(vif_q2_6_partes, ignore_index=True) if vif_q2_6_partes else pd.DataFrame()

  (!) Variables sin varianza excluidas del VIF: ['fundamental_frecuencia_efectiva_90d']


BTCUSD: 29/54 variables con VIF > 10.
  (!) Variables 100% nulas excluidas de la regresión (Close): ['x2_score', 'x2_score_cross', 'x2_score_tendencia', 'fundamental_cambio', 'fundamental_direccion_cambio', 'fundamental_magnitud_cambio', 'fundamental_estabilidad_90d', 'fundamental_contradiccion']


  (!) Columnas sin varianza (constantes) descartadas: ['fundamental_frecuencia_efectiva_90d']


  (!) Variables 100% nulas excluidas de la regresión (Close): ['x2_score', 'x2_score_cross', 'x2_score_tendencia', 'fundamental_cambio', 'fundamental_direccion_cambio', 'fundamental_magnitud_cambio', 'fundamental_estabilidad_90d', 'fundamental_contradiccion']
  (!) Columnas sin varianza (constantes) descartadas: ['fundamental_frecuencia_efectiva_90d']
BTCUSD/Close: R²=0.970 (n=39440), 7 variable(s) cambian de signo al pasar a la especificación reducida.
  (!) Variables 100% nulas excluidas de la regresión (retorno_log_1h): ['x2_score', 'x2_score_cross', 'x2_score_tendencia', 'fundamental_cambio', 'fundamental_direccion_cambio', 'fundamental_magnitud_cambio', 'fundamental_estabilidad_90d', 'fundamental_contradiccion']
  (!) Columnas sin varianza (constantes) descartadas: ['fundamental_frecuencia_efectiva_90d']
  (!) Variables 100% nulas excluidas de la regresión (retorno_log_1h): ['x2_score', 'x2_score_cross', 'x2_score_tendencia', 'fundamental_cambio', 'fundamental_direccion_cambio', '

### 7.1. Ranking cruzado con el Pearson crudo de Q2-3

Cruza el coeficiente ceteris paribus (especificación completa) con el
Pearson crudo de la Sección 3 (`ranking_q2_3`) — identifica qué variables
mantienen signo al controlar por las demás (`coincide_signo_con_pearson`) y
cuáles lo invierten.

In [24]:
ranking_q2_6 = (
    ceteris_q2_6
    .rename(columns={'feature': 'variable'})
    .merge(ranking_q2_3[['valor', 'variable', 'target', 'pearson']], on=['valor', 'variable', 'target'], how='left')
    .assign(coincide_signo_con_pearson=lambda d: np.sign(d['coef_ceteris_paribus']) == np.sign(d['pearson']))
    .assign(abs_coef=lambda d: d['coef_ceteris_paribus'].abs())
    .sort_values(['valor', 'target', 'abs_coef'], ascending=[True, True, False])
    .drop(columns='abs_coef')
    .reset_index(drop=True)
)
ranking_q2_6

,variable,coef_ceteris_paribus,error_estandar,t,p_valor,significativo_5pct,coef_reducida,cambia_signo_al_reducir,valor,target,vif,pearson,coincide_signo_con_pearson
0,duracion_drawdown,-0.817577,0.005567,-146.872980,0.000000,True,NaN,False,BTCUSD,Close,40.722929,-0.659814,True
1,recuperacion_desde_minimo,0.409235,0.003493,117.169949,0.000000,True,NaN,False,BTCUSD,Close,16.031654,-0.352564,False
2,fundamental_edad,0.371776,0.001696,219.186367,0.000000,True,0.334296,False,BTCUSD,Close,3.780954,0.792445,True
3,frecuencia_extremos_30d,-0.237635,0.002738,-86.804280,0.000000,True,-0.127576,False,BTCUSD,Close,9.849288,-0.042660,True
4,drawdown_365d,0.181599,0.002725,66.649306,0.000000,True,0.183917,False,BTCUSD,Close,9.756622,0.574979,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,cambio_volatilidad,-0.001060,0.007988,-0.132733,0.894405,False,0.017174,True,BTCUSD,retorno_log_1h,8.508987,0.001305,False
104,pendiente_sma_200,-0.000411,0.004061,-0.101252,0.919351,False,-0.017122,False,BTCUSD,retorno_log_1h,2.199249,0.003641,False
105,frecuencia_extremos_30d,-0.000120,0.008594,-0.013959,0.988863,False,-0.018885,False,BTCUSD,retorno_log_1h,9.849288,0.001613,False
106,fundamental_edad,-0.000112,0.005325,-0.021009,0.983239,False,-0.003192,False,BTCUSD,retorno_log_1h,3.780954,0.003810,False


### 7.2. Guardar el ceteris paribus y el VIF

Mismo criterio de un CSV por ítem que las secciones anteriores — para que la
celda de export a zip del final del notebook (`Q2-5`/`Q2-EXEC`) los incluya.

In [25]:
for valor in tablas:
    path_ranking = CARPETA_EXP / f'{valor}_q2_6_ceteris_paribus.csv'
    path_vif = CARPETA_EXP / f'{valor}_q2_6_vif.csv'
    ranking_q2_6[ranking_q2_6['valor'] == valor].to_csv(path_ranking, index=False)
    vif_q2_6[vif_q2_6['valor'] == valor].to_csv(path_vif, index=False)
    print(f"{valor}: ceteris paribus guardado en {path_ranking.name}, VIF en {path_vif.name}")

BTCUSD: ceteris paribus guardado en BTCUSD_q2_6_ceteris_paribus.csv, VIF en BTCUSD_q2_6_vif.csv


## 8. Estabilidad histórica de las relaciones (Q2-9)

**Objetivo**: evaluar si la asociación variable↔target (Secciones 3/4,
`Q2-3`/`Q2-4`) es estable en el tiempo o cambia entre períodos — una relación
fuerte en el Pearson global puede estar promediando sub-períodos con signos
opuestos.

**Pregunta**: ¿la relación de cada variable con el precio/retorno se
mantiene igual a lo largo de los años del histórico, o hay períodos donde se
invierte o desaparece?

**Decisión de diseño — mismas variables/targets que Q2-3/Q2-4**: se reutilizan
`_variables_analisis_q2_3`, `TARGETS_Q2_3` y la función `coseno` de la
Sección 4 (`Q2-4`) — no se introduce un criterio de selección nuevo.

**Metodología**:
- **Por año calendario** (`DateTime.dt.year`): Pearson, Spearman y coseno
  calculados por separado dentro de cada año — la partición temporal más
  simple e interpretable, alineada con régimen de mercado (ej. 2022 bajista,
  2023-2024 alcista).
- **Rolling simple** (ventana de `VENTANA_ROLLING_DIAS=90` días en velas H1):
  Pearson rolling entre la variable y el target, resumido como su desviación
  estándar en el tiempo — una ventana más fina que el año, pero solo para
  Pearson (no coseno); el rolling específico de coseno con comparación de
  ventanas es responsabilidad de la Sección 9 (`Q2-10`), para no duplicar
  análisis.
- **Resumen de estabilidad**: por variable/target, si el signo del Pearson se
  mantiene igual en todos los años y qué tan dispersos están sus valores
  entre años (desviación estándar) — `estable` si todos los años coinciden en
  signo y la dispersión es baja (`< UMBRAL_STD_INESTABLE`), `inestable` si hay
  dispersión alta sin cambio de signo, `cambio estructural` si el signo
  cambia entre años.

**Interpretación**: una variable `estable` es más confiable para pensar
configuración (Parte III) que una `cambio estructural`, aunque esta última
tenga el Pearson global más alto — el global puede estar promediando dos
relaciones opuestas en distintos regímenes.

**Limitaciones**: la partición por año es arbitraria (no coincide
necesariamente con los puntos reales de cambio de régimen); con pocos años de
historia (BTCUSD parte en 2021) cada "año" tiene distinta cantidad de
observaciones y el primer/último año del histórico suelen estar
incompletos — se excluyen de la clasificación los años con menos de 30
observaciones no nulas, igual mínimo que el resto del notebook.

In [26]:
VENTANA_ROLLING_DIAS = 90
VENTANA_ROLLING_H = VENTANA_ROLLING_DIAS * 24


def _pearson_spearman_coseno(sub, col, target_col):
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None
    pear_r, _ = scipy_stats.pearsonr(x, y)
    spear_r, _ = scipy_stats.spearmanr(x, y)
    return {'n': len(sub), 'pearson': pear_r, 'spearman': spear_r, 'coseno': coseno(x, y)}


resultados_q2_9_anio = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    anios = sorted(tabla['DateTime'].dt.year.unique())
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            for anio in anios:
                sub = tabla.loc[tabla['DateTime'].dt.year == anio, [col, target_col]].dropna()
                metricas = _pearson_spearman_coseno(sub, col, target_col)
                if metricas is None:
                    continue
                resultados_q2_9_anio.append({'valor': valor, 'variable': col, 'target': target_col,
                                              'anio': int(anio), **metricas})

por_anio_q2_9 = pd.DataFrame(resultados_q2_9_anio)
print(f"Filas año x variable x target generadas: {len(por_anio_q2_9)} (años: {anios})")

Filas año x variable x target generadas: 648 (años: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)])


In [27]:
UMBRAL_STD_INESTABLE = 0.15  # std del Pearson entre años a partir de la cual se considera dispersión alta


def _clasificar_estabilidad(pearsons_por_anio):
    signos = set(np.sign(pearsons_por_anio)[np.sign(pearsons_por_anio) != 0])
    if len(signos) > 1:
        return 'cambio estructural'
    return 'inestable' if pearsons_por_anio.std() > UMBRAL_STD_INESTABLE else 'estable'


resumen_q2_9 = []
for (valor, variable, target_col), grupo in por_anio_q2_9.groupby(['valor', 'variable', 'target']):
    pearson_global = ranking_q2_7.loc[
        (ranking_q2_7['valor'] == valor) & (ranking_q2_7['variable'] == variable) & (ranking_q2_7['target'] == target_col),
        'pearson'
    ]
    resumen_q2_9.append({
        'valor': valor, 'variable': variable, 'target': target_col,
        'n_anios': len(grupo),
        'pearson_global': pearson_global.iloc[0] if len(pearson_global) else np.nan,
        'pearson_std_entre_anios': grupo['pearson'].std(),
        'estabilidad': _clasificar_estabilidad(grupo['pearson'].to_numpy()),
    })

estabilidad_q2_9 = pd.DataFrame(resumen_q2_9)
print(estabilidad_q2_9['estabilidad'].value_counts().to_string())

estabilidad
estable               55
cambio estructural    50
inestable              3


In [28]:
resultados_q2_9_rolling = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            sub = tabla[[col, target_col]].dropna()
            if len(sub) < VENTANA_ROLLING_H * 2:
                continue
            rolling_corr = sub[col].rolling(VENTANA_ROLLING_H, min_periods=VENTANA_ROLLING_H // 2).corr(sub[target_col])
            rolling_corr = rolling_corr.dropna()
            if rolling_corr.empty:
                continue
            resultados_q2_9_rolling.append({
                'valor': valor, 'variable': col, 'target': target_col,
                'pearson_rolling_std': rolling_corr.std(),
                'pearson_rolling_min': rolling_corr.min(),
                'pearson_rolling_max': rolling_corr.max(),
            })

rolling_q2_9 = pd.DataFrame(resultados_q2_9_rolling)
estabilidad_q2_9 = estabilidad_q2_9.merge(rolling_q2_9, on=['valor', 'variable', 'target'], how='left')
estabilidad_q2_9 = estabilidad_q2_9.sort_values(['valor', 'target', 'pearson_std_entre_anios'],
                                                 ascending=[True, True, False]).reset_index(drop=True)
estabilidad_q2_9

/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:51: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/opt/anaconda3/envs/revenAI/lib/python3.11/site-packages/numpy/_core/_methods.py:51: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)


,valor,variable,target,n_anios,pearson_global,pearson_std_entre_anios,estabilidad,pearson_rolling_std,pearson_rolling_min,pearson_rolling_max
0,BTCUSD,fundamental_edad,Close,6,0.792445,0.774796,cambio estructural,0.684903,-0.959798,0.970019
1,BTCUSD,duracion_drawdown,Close,6,-0.659814,0.645451,cambio estructural,0.611694,-0.959798,0.970019
2,BTCUSD,recuperacion_desde_minimo,Close,6,-0.352564,0.590136,cambio estructural,0.596764,-0.842339,1.000000
3,BTCUSD,velocidad_caida,Close,6,-0.167871,0.545647,cambio estructural,0.522766,-0.733714,0.999011
4,BTCUSD,vol_alcista_30d,Close,6,-0.341128,0.492622,cambio estructural,0.491595,-0.918287,0.933665
...,...,...,...,...,...,...,...,...,...,...
103,BTCUSD,x3_roc_10,retorno_log_1h,6,0.316032,0.004479,estable,0.010840,0.277920,0.361539
104,BTCUSD,x3_sma_dist_200,retorno_log_1h,6,0.122750,0.004173,estable,0.006123,0.044148,0.193410
105,BTCUSD,x3_sma_dist_20,retorno_log_1h,6,0.383103,0.003196,estable,0.004389,0.340787,0.426030
106,BTCUSD,x3_ema_dist_26,retorno_log_1h,6,0.377274,0.001682,estable,0.002806,0.321800,0.421179


### 8.1. Guardar las métricas por año y el resumen de estabilidad

Mismo criterio que las secciones anteriores — para que la celda de export a
zip del final del notebook (`Q2-5`/`Q2-EXEC`) los incluya.

In [29]:
for valor in tablas:
    path_anio = CARPETA_EXP / f'{valor}_q2_9_por_anio.csv'
    path_estabilidad = CARPETA_EXP / f'{valor}_q2_9_estabilidad.csv'
    por_anio_q2_9[por_anio_q2_9['valor'] == valor].to_csv(path_anio, index=False)
    estabilidad_q2_9[estabilidad_q2_9['valor'] == valor].to_csv(path_estabilidad, index=False)
    print(f"{valor}: métricas por año guardadas en {path_anio.name}, resumen de estabilidad en {path_estabilidad.name}")

BTCUSD: métricas por año guardadas en BTCUSD_q2_9_por_anio.csv, resumen de estabilidad en BTCUSD_q2_9_estabilidad.csv


## 9. Coseno rolling por ventanas (Q2-10)

**Objetivo**: evaluar si la alineación direccional `cos(Pv,Xv)` (Sección 4,
`Q2-4`) es estable en el tiempo, de forma análoga a cómo la Sección 8
(`Q2-9`) evalúa la estabilidad de Pearson — pero para coseno, y comparando
explícitamente varios tamaños de ventana en vez de depender de uno solo
(razón del nombre "por ventanas" del ítem en `docs/tracking/todos.md`).

**Pregunta**: ¿la alineación direccional entre cada variable y el target se
mantiene en el tiempo, o su signo/magnitud cambia según el período — y esa
conclusión depende de qué tamaño de ventana se elige para mirarla?

**Decisión de diseño — mismas variables/targets que Q2-3/Q2-4/Q2-9, ventanas
nuevas**: se reutilizan `_variables_analisis_q2_3`, `TARGETS_Q2_3` y la
función `coseno` ya definida en la Sección 4 — no se introduce un criterio de
selección nuevo. Lo nuevo es `VENTANAS_ROLLING_DIAS_Q2_10 = [30, 90, 180]`
días (en velas H1): incluye la misma ventana de 90 días que usa el Pearson
rolling de `Q2-9` (comparable directamente con esa sección) más una ventana
más corta (30d) y una más larga (180d), para ver si la lectura de estabilidad
cambia según la escala temporal elegida.

**Metodología**: no existe un equivalente a `.rolling().corr()` para coseno
en pandas, así que el coseno rolling se calcula de forma vectorizada: suma
rolling de `x·y`, `x²` y `y²` sobre cada ventana, de donde sale
`cos = Σ(x·y) / (√Σx² · √Σy²)`. Por cada ventana se resume la serie rolling
resultante con su desvío estándar, mínimo y máximo (mismo resumen que
`Q2-9`), y se clasifica con el mismo criterio ya usado ahí: `cambio
estructural` si el rolling cruza cero (mínimo y máximo de signo distinto),
`inestable` si no cruza cero pero su dispersión supera `UMBRAL_STD_INESTABLE`
(heredado de `Q2-9`), `estable` en otro caso. Finalmente se agrega la
pregunta nueva de esta sección: si esa clasificación es la misma en las 3
ventanas (`consistente_entre_ventanas`) — si no lo es, la "estabilidad" de
esa variable depende de una elección arbitraria de ventana y debe leerse con
cautela.

**Interpretación**: una variable `consistente_entre_ventanas=True` es una
señal más confiable para la síntesis (`Q2-11`) que una cuya clasificación
cambia según la ventana — en ese caso conviene mirar el detalle por ventana
(`rolling_q2_10`) antes de usarla para una hipótesis de parámetro.

**Limitaciones**: mismas de `Q2-9` (partición temporal simplificada, no
necesariamente alineada con cambios reales de régimen; autocorrelación);
con la ventana de 180 días se pierden más observaciones al inicio de la
serie que con 30/90, así que algunas variables pueden tener menos ventanas
válidas que otras.

In [30]:
VENTANAS_ROLLING_DIAS_Q2_10 = [30, 90, 180]  # incluye los 90d de Q2-9 para comparar directo con su Pearson rolling


def _coseno_rolling(sub, col, target_col, ventana_h):
    """Coseno rolling vectorizado: no existe equivalente a .rolling().corr() para coseno."""
    x, y = sub[col], sub[target_col]
    min_periods = ventana_h // 2
    dot = (x * y).rolling(ventana_h, min_periods=min_periods).sum()
    norm_x = np.sqrt((x ** 2).rolling(ventana_h, min_periods=min_periods).sum())
    norm_y = np.sqrt((y ** 2).rolling(ventana_h, min_periods=min_periods).sum())
    denom = (norm_x * norm_y).replace(0, np.nan)
    return (dot / denom).dropna()


resultados_q2_10 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            sub = tabla[[col, target_col]].dropna()
            for dias in VENTANAS_ROLLING_DIAS_Q2_10:
                ventana_h = dias * 24
                if len(sub) < ventana_h * 2:
                    continue
                serie = _coseno_rolling(sub, col, target_col, ventana_h)
                if serie.empty:
                    continue
                resultados_q2_10.append({
                    'valor': valor, 'variable': col, 'target': target_col, 'ventana_dias': dias,
                    'coseno_rolling_std': serie.std(), 'coseno_rolling_min': serie.min(), 'coseno_rolling_max': serie.max(),
                })

rolling_q2_10 = pd.DataFrame(resultados_q2_10)
print(f"Filas variable x target x ventana generadas: {len(rolling_q2_10)} (ventanas: {VENTANAS_ROLLING_DIAS_Q2_10} días)")

Filas variable x target x ventana generadas: 324 (ventanas: [30, 90, 180] días)


### 9.1. Clasificación de estabilidad por ventana y consistencia entre ventanas

Mismo criterio de clasificación que `Q2-9` (`UMBRAL_STD_INESTABLE`), aplicado
ahora por ventana, más la pregunta nueva: ¿la clasificación resultante es la
misma en las 3 ventanas?

In [31]:
def _clasificar_estabilidad_rolling(minimo, maximo, std, umbral_std=UMBRAL_STD_INESTABLE):
    """Mismo criterio de Q2-9: cruce de signo entre mínimo y máximo -> cambio estructural; si no, umbral de dispersión."""
    if minimo != 0 and maximo != 0 and np.sign(minimo) != np.sign(maximo):
        return 'cambio estructural'
    return 'inestable' if std > umbral_std else 'estable'


rolling_q2_10['clasificacion'] = rolling_q2_10.apply(
    lambda r: _clasificar_estabilidad_rolling(r['coseno_rolling_min'], r['coseno_rolling_max'], r['coseno_rolling_std']),
    axis=1,
)

resumen_q2_10 = []
for (valor, variable, target_col), grupo in rolling_q2_10.groupby(['valor', 'variable', 'target']):
    clasificaciones = grupo['clasificacion']
    resumen_q2_10.append({
        'valor': valor, 'variable': variable, 'target': target_col,
        'n_ventanas': len(grupo),
        'clasificacion_moda': clasificaciones.mode().iat[0],
        'consistente_entre_ventanas': clasificaciones.nunique() == 1,
    })

resumen_q2_10 = pd.DataFrame(resumen_q2_10).sort_values(['valor', 'target', 'consistente_entre_ventanas']).reset_index(drop=True)
print(f"Variables con clasificación consistente en las {len(VENTANAS_ROLLING_DIAS_Q2_10)} ventanas: "
      f"{int(resumen_q2_10['consistente_entre_ventanas'].sum())}/{len(resumen_q2_10)}")
resumen_q2_10

Variables con clasificación consistente en las 3 ventanas: 96/108


,valor,variable,target,n_ventanas,clasificacion_moda,consistente_entre_ventanas
0,BTCUSD,duracion_drawdown,Close,3,estable,False
1,BTCUSD,recuperacion_desde_minimo,Close,3,estable,False
2,BTCUSD,ruido_atr_sin_direccion_30d,Close,3,estable,False
3,BTCUSD,ruido_ratio_movimiento_30d,Close,3,estable,False
4,BTCUSD,adx_14,Close,3,estable,True
...,...,...,...,...,...,...
103,BTCUSD,x3_sma_dist_20,retorno_log_1h,3,estable,True
104,BTCUSD,x3_sma_dist_50,retorno_log_1h,3,estable,True
105,BTCUSD,x3_trend_slope_50,retorno_log_1h,3,cambio estructural,True
106,BTCUSD,x3_vol_24h,retorno_log_1h,3,cambio estructural,True


### 9.2. Guardar el coseno rolling por ventana y el resumen de consistencia

Mismo criterio que las secciones anteriores — para que la celda de export a
zip del final del notebook (`Q2-5`/`Q2-EXEC`) los incluya.

In [32]:
for valor in tablas:
    path_rolling = CARPETA_EXP / f'{valor}_q2_10_rolling.csv'
    path_resumen = CARPETA_EXP / f'{valor}_q2_10_resumen.csv'
    rolling_q2_10[rolling_q2_10['valor'] == valor].to_csv(path_rolling, index=False)
    resumen_q2_10[resumen_q2_10['valor'] == valor].to_csv(path_resumen, index=False)
    print(f"{valor}: coseno rolling por ventana guardado en {path_rolling.name}, resumen en {path_resumen.name}")

BTCUSD: coseno rolling por ventana guardado en BTCUSD_q2_10_rolling.csv, resumen en BTCUSD_q2_10_resumen.csv


## 10. Relaciones entre variables y redundancia (Q2-12)

**Objetivo**: a diferencia de todas las secciones anteriores (variable ↔
target), esta mide relaciones ENTRE las propias variables explicativas —
técnico vs. técnico, fundamental vs. fundamental y técnico vs. fundamental —
para identificar pares que aportan prácticamente la misma información.

**Pregunta**: ¿qué pares de variables están tan correlacionados entre sí que,
para efectos prácticos, son la misma señal contada dos veces?

**Familia técnico/fundamental**: no existe una columna de familia explícita
en el catálogo de `X5_Q1.ipynb` (`{valor}_catalogo_variables.csv`), así que se
deriva de su columna `fuente` — mapeo explícito, documentado acá para que sea
auditable: `técnico` = `X3 (técnico)`, `Q1 — Técnico transformado`,
`Q1 — Drawdown`, `Q1 — Volatilidad`, `Q1 — Liquidez`, `Q1 — Ruido`,
`Q1 — Tendencia/régimen` (todo lo construido a partir del precio);
`fundamental` = `X2 (fundamental)`, `Q1 — Fundamental (estado)`. Las columnas
de precio crudo (`fuente = 'Precio (MT5)'`) ya están excluidas por
`_variables_analisis_q2_3`, igual que en el resto del notebook.

**Metodología**: matriz de correlación de Pearson entre todas las variables
de `_variables_analisis_q2_3` (mismo set que el resto de Q2); cada par único
`(var_a, var_b)` se clasifica en uno de los 3 grupos de familia de arriba y
se marca `redundante` si `|Pearson| > UMBRAL_REDUNDANCIA_Q2_12 = 0.85` — un
corte más estricto que el `VIF > 10` de `Q2-6`, porque acá se mide la
similitud bivariada directa entre dos variables, no el efecto conjunto de
todas las demás sobre una (lo que ya hace `Q2-6`). Se excluyen antes
de calcular la matriz las variables sin ningún dato o sin varianza en este
snapshot de Mac (`x2_score` y derivadas, `fundamental_cambio` y derivadas —
misma limitación ya documentada en `[Q1-10]`/`[Q2-2]`/`[Q2-6]` —, 9 de las 63
variables de `_variables_analisis_q2_3`), con aviso explícito: su
correlación con cualquier otra variable queda indefinida (`NaN`), no cero.

**Clustering/PCA — se omite por ahora**: el TODO los menciona como opcional
("solo si aporta interpretabilidad"). El conteo y listado de pares
redundantes por grupo de familia (Sección 10.1) ya es interpretable por sí
solo para esta primera pasada — ir de menos a más: si más adelante aparecen
grupos con muchos pares redundantes simultáneos y resulta difícil de leer
como lista plana, ahí se justifica agregar PCA/clustering, no antes.

**Interpretación**: un par marcado `redundante` no implica que haya que
eliminar una de las dos variables — solo que, de usarse ambas como variables
explicativas de un parámetro de X5 (Parte III, `Q2-11`), aportan señal muy
parecida; dos variables con Pearson alto entre sí pero relación distinta con
el target (`ranking_q2_7`) pueden seguir siendo ambas útiles.

**Limitaciones**: Pearson entre variables no captura relaciones no lineales
entre ellas (mismo matiz de linealidad que el resto del notebook); con ~63
variables el total de pares (~1.950) hace que incluso relaciones moderadas
aparezcan muchas veces — el foco de esta sección es la cola de pares con
correlación muy alta, no una lectura exhaustiva de todos los pares.

In [33]:
UMBRAL_REDUNDANCIA_Q2_12 = 0.85  # |Pearson| bivariado a partir del cual dos variables se consideran "casi la misma información" (más estricto que el VIF>10 de Q2-6, que mide colinealidad conjunta)

FUENTES_TECNICO_Q2_12 = {
    'X3 (técnico)', 'Q1 — Técnico transformado', 'Q1 — Drawdown',
    'Q1 — Volatilidad', 'Q1 — Liquidez', 'Q1 — Ruido', 'Q1 — Tendencia/régimen',
}
FUENTES_FUNDAMENTAL_Q2_12 = {'X2 (fundamental)', 'Q1 — Fundamental (estado)'}

GRUPOS_FAMILIA_Q2_12 = {
    frozenset({'técnico'}): 'técnico-técnico',
    frozenset({'fundamental'}): 'fundamental-fundamental',
    frozenset({'técnico', 'fundamental'}): 'técnico-fundamental',
}


def _cargar_familia_q2_12(valor):
    """Deriva la familia técnico/fundamental de cada variable desde la columna `fuente`
    del catálogo de X5_Q1 — no hay una columna de familia explícita todavía."""
    catalogo = pd.read_csv(carpeta_exp('X5_Q1') / f'{valor}_catalogo_variables.csv')
    familia = {}
    for _, fila in catalogo.iterrows():
        if fila['fuente'] in FUENTES_TECNICO_Q2_12:
            familia[fila['variable']] = 'técnico'
        elif fila['fuente'] in FUENTES_FUNDAMENTAL_Q2_12:
            familia[fila['variable']] = 'fundamental'
    return familia


resultados_q2_12 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    familia = _cargar_familia_q2_12(valor)
    variables_con_familia = [v for v in variables_analisis if v in familia]
    sin_familia = sorted(set(variables_analisis) - set(familia))
    if sin_familia:
        print(f"{valor}: (!) variables sin familia técnico/fundamental en el catálogo, excluidas de Q2-12: {sin_familia}")

    sin_varianza = [v for v in variables_con_familia if tabla[v].std(skipna=True) in (0, None) or pd.isna(tabla[v].std(skipna=True))]
    if sin_varianza:
        print(f"{valor}: (!) variables sin dato o sin varianza excluidas de Q2-12 (correlación indefinida): {sin_varianza}")
    variables_con_familia = [v for v in variables_con_familia if v not in sin_varianza]

    corr = tabla[variables_con_familia].corr(numeric_only=True)
    for i, var_a in enumerate(variables_con_familia):
        for var_b in variables_con_familia[i + 1:]:
            r = corr.loc[var_a, var_b]
            if pd.isna(r):
                continue
            grupo = GRUPOS_FAMILIA_Q2_12[frozenset({familia[var_a], familia[var_b]})]
            resultados_q2_12.append({
                'valor': valor, 'var_a': var_a, 'var_b': var_b,
                'familia_a': familia[var_a], 'familia_b': familia[var_b], 'grupo_familia': grupo,
                'pearson': r, 'redundante': bool(abs(r) > UMBRAL_REDUNDANCIA_Q2_12),
            })

pares_q2_12 = pd.DataFrame(resultados_q2_12)
print(f"Pares evaluados: {len(pares_q2_12)}; redundantes (|Pearson|>{UMBRAL_REDUNDANCIA_Q2_12}): {int(pares_q2_12['redundante'].sum())}")

BTCUSD: (!) variables sin dato o sin varianza excluidas de Q2-12 (correlación indefinida): ['x2_score', 'x2_score_cross', 'x2_score_tendencia', 'fundamental_cambio', 'fundamental_direccion_cambio', 'fundamental_magnitud_cambio', 'fundamental_frecuencia_efectiva_90d', 'fundamental_estabilidad_90d', 'fundamental_contradiccion']
Pares evaluados: 1431; redundantes (|Pearson|>0.85): 46


### 10.1. Resumen por grupo de familia y pares redundantes

In [34]:
resumen_q2_12 = (
    pares_q2_12
    .groupby(['valor', 'grupo_familia'])
    .agg(n_pares=('pearson', 'size'), n_redundantes=('redundante', 'sum'))
    .reset_index()
)
resumen_q2_12['pct_redundantes'] = (resumen_q2_12['n_redundantes'] / resumen_q2_12['n_pares'] * 100).round(1)
resumen_q2_12

,valor,grupo_familia,n_pares,n_redundantes,pct_redundantes
0,BTCUSD,técnico-fundamental,53,0,0.0
1,BTCUSD,técnico-técnico,1378,46,3.3


In [35]:
pares_redundantes_q2_12 = (
    pares_q2_12[pares_q2_12['redundante']]
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'abs_pearson'], ascending=[True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
print(f"Pares redundantes listados: {len(pares_redundantes_q2_12)}")
pares_redundantes_q2_12

Pares redundantes listados: 46


,valor,var_a,var_b,familia_a,familia_b,grupo_familia,pearson,redundante
0,BTCUSD,ruido_ratio_movimiento_30d,ruido_atr_sin_direccion_30d,técnico,técnico,técnico-técnico,0.999931,True
1,BTCUSD,x3_rsi_14,distancia_ema_26,técnico,técnico,técnico-técnico,0.991118,True
2,BTCUSD,x3_sma_dist_20,x3_ema_dist_26,técnico,técnico,técnico-técnico,0.964192,True
3,BTCUSD,distancia_sma_20,distancia_ema_26,técnico,técnico,técnico-técnico,0.961763,True
4,BTCUSD,x3_sma_dist_20,x3_ema_dist_12,técnico,técnico,técnico-técnico,0.957629,True
5,BTCUSD,distancia_sma_20,distancia_ema_12,técnico,técnico,técnico-técnico,0.955993,True
6,BTCUSD,x3_macd,x3_macd_signal,técnico,técnico,técnico-técnico,0.951418,True
7,BTCUSD,pendiente_sma_20,pendiente_ema_26,técnico,técnico,técnico-técnico,0.945351,True
8,BTCUSD,x3_rsi_14,distancia_sma_20,técnico,técnico,técnico-técnico,0.945164,True
9,BTCUSD,x3_rsi_14,distancia_sma_50,técnico,técnico,técnico-técnico,0.934310,True


### 10.2. Guardar los pares, el resumen por familia y los pares redundantes

Mismo criterio que las secciones anteriores — para que la celda de export a
zip del final del notebook (`Q2-5`/`Q2-EXEC`) los incluya.

In [36]:
for valor in tablas:
    path_pares = CARPETA_EXP / f'{valor}_q2_12_pares.csv'
    path_resumen = CARPETA_EXP / f'{valor}_q2_12_resumen.csv'
    path_redundantes = CARPETA_EXP / f'{valor}_q2_12_redundantes.csv'
    pares_q2_12[pares_q2_12['valor'] == valor].to_csv(path_pares, index=False)
    resumen_q2_12[resumen_q2_12['valor'] == valor].to_csv(path_resumen, index=False)
    pares_redundantes_q2_12[pares_redundantes_q2_12['valor'] == valor].to_csv(path_redundantes, index=False)
    print(f"{valor}: pares guardados en {path_pares.name}, resumen en {path_resumen.name}, redundantes en {path_redundantes.name}")

BTCUSD: pares guardados en BTCUSD_q2_12_pares.csv, resumen en BTCUSD_q2_12_resumen.csv, redundantes en BTCUSD_q2_12_redundantes.csv


## 11. KPIs adicionales de Data Science (Q2-14)

**Objetivo**: el TODO (`docs/tracking/todos.md`) pide evaluar *caso a caso* —
no adoptar por default — si una métrica adicional de asociación aporta algo
que Pearson/Spearman/coseno (Secciones 3/4) no estén ya mostrando. Se eligen
dos candidatas señaladas ahí mismo:

- **Información mutua** (`mutual_info_regression`, `sklearn` — ya parte del
  stack del proyecto, sin agregar dependencia nueva): captura cualquier
  dependencia estadística, lineal o no, entre variable y target.
- **Correlación de distancia** (Székely–Rizzo): también captura cualquier
  forma de dependencia, pero a diferencia de la información mutua queda
  acotada en `[0,1]` — comparable directamente con `|Pearson|`.

**Pregunta**: ¿hay variables donde información mutua o correlación de
distancia revelan una asociación relevante con el target que Pearson/
Spearman no están capturando (porque la relación no es lineal ni monótona)?

**Decisión de diseño — mismas variables/targets que Q2-3/Q2-4**: se reutiliza
`_variables_analisis_q2_3` y `TARGETS_Q2_3` — no se introduce un criterio de
selección nuevo.

**Metodología**:
- **Información mutua**: `mutual_info_regression` sobre todas las filas no
  nulas de cada par variable/target (rápido — no es `O(n²)`), con
  `n_neighbors=3` (default razonable del estimador por vecinos más cercanos).
  Se reporta en nats (no acotado en `[0,1]`), así que la comparación con
  Pearson/Spearman se hace por **ranking**, no por valor absoluto: se marca
  `entra_top_mi_sin_estar_en_pearson` cuando una variable entra al
  top-`TOP_N_KPI_Q2_14=10` por información mutua sin estar en el top-10 por
  `|Pearson|` — candidata a relación no lineal no capturada por el ranking
  lineal.
- **Correlación de distancia**: `O(n²)` en tiempo y memoria (matrices de
  distancia completas) — inviable sobre las ~40.000 filas de la tabla
  completa, así que se calcula sobre una **submuestra fija**
  (`N_MUESTRA_DCOR_Q2_14=3000` filas, semilla fija `42`, la misma submuestra
  para todas las variables/targets de un mismo `valor`) implementada a mano
  con `numpy` (doble centrado de las matrices de distancia — no se agrega
  `dcor` como dependencia nueva para esto). Al estar acotada en `[0,1]`, se
  compara directo contra `|Pearson|`: se marca `aporta_valor_dcor` cuando
  `distance_corr - |Pearson| > UMBRAL_DIFERENCIA_DCOR_Q2_14=0.10`.

**No se agregan más KPIs que estos dos** — el TODO advierte explícitamente
contra sobrecargar el notebook con técnicas que no respondan una pregunta
concreta; información mutua y correlación de distancia ya cubren, entre las
dos, tanto la comparación acotada directa (dcor) como la comparación por
ranking sin asumir una escala común (MI). No se evalúan más candidatas en
esta pasada.

**Interpretación**: estas dos columnas son un filtro de atención, no un
reemplazo de Pearson/Spearman/coseno — si ninguna variable queda marcada en
`aporta_valor_dcor` ni en `entra_top_mi_sin_estar_en_pearson`, la conclusión
es que, para esta tabla, Pearson/Spearman/coseno ya capturan razonablemente
bien las relaciones existentes y no hace falta incorporar estas métricas a
la síntesis (`Q2-11`) ni a las hipótesis de parámetros (`Q2-11` tabla final,
`X5_Q3.ipynb`).

**Limitaciones**: `mutual_info_regression` asume continuidad en ambas
variables — con columnas discretas/binarias de la tabla (`ruptura_rango`,
`duracion_en_rango`, etc.) el estimador por vecinos más cercanos puede ser
menos preciso, pero se aplica la misma función a todas las variables por
simplicidad, igual criterio que el resto del notebook; la submuestra de
`dcor` introduce varianza de muestreo adicional frente a un cálculo sobre el
dataset completo (inviable aquí por costo computacional).

In [37]:
from sklearn.feature_selection import mutual_info_regression

N_MUESTRA_DCOR_Q2_14 = 3000  # dcor es O(n²) en tiempo/memoria — inviable sobre las ~40.000 filas completas
UMBRAL_DIFERENCIA_DCOR_Q2_14 = 0.10  # dcor por sobre |Pearson| en más de este margen -> candidata a relación no lineal no capturada por Pearson/Spearman
TOP_N_KPI_Q2_14 = 10  # tamaño del top usado para comparar el ranking de información mutua contra el de |Pearson|


def _distance_correlation(x, y):
    """Correlación de distancia (Székely-Rizzo): capta cualquier forma de dependencia, no solo lineal (Pearson)
    o monótona (Spearman). O(n²) en tiempo/memoria — se calcula sobre una submuestra fija, nunca sobre la tabla completa."""
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=float).reshape(-1, 1)
    a, b = np.abs(x - x.T), np.abs(y - y.T)
    A = a - a.mean(axis=0, keepdims=True) - a.mean(axis=1, keepdims=True) + a.mean()
    B = b - b.mean(axis=0, keepdims=True) - b.mean(axis=1, keepdims=True) + b.mean()
    dcov2, dvarx2, dvary2 = (A * B).mean(), (A * A).mean(), (B * B).mean()
    denom = np.sqrt(dvarx2 * dvary2)
    return float(np.sqrt(max(dcov2, 0) / denom)) if denom > 0 else np.nan

In [38]:
rng_q2_14 = np.random.default_rng(42)
resultados_q2_14 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    idx_muestra = rng_q2_14.choice(tabla.index, size=min(N_MUESTRA_DCOR_Q2_14, len(tabla)), replace=False)
    muestra = tabla.loc[idx_muestra]
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            sub_full = tabla[[col, target_col]].dropna()
            if len(sub_full) < 30 or sub_full[col].std() == 0 or sub_full[target_col].std() == 0:
                continue
            mi = mutual_info_regression(
                sub_full[[col]].to_numpy(), sub_full[target_col].to_numpy(), random_state=42, n_neighbors=3,
            )[0]

            sub_muestra = muestra[[col, target_col]].dropna()
            dcor = (
                _distance_correlation(sub_muestra[col].to_numpy(), sub_muestra[target_col].to_numpy())
                if len(sub_muestra) >= 30 and sub_muestra[col].std() > 0 and sub_muestra[target_col].std() > 0
                else np.nan
            )
            resultados_q2_14.append({
                'valor': valor, 'variable': col, 'target': target_col,
                'n_mi': len(sub_full), 'mutual_info': mi,
                'n_dcor': len(sub_muestra), 'distance_corr': dcor,
            })
    print(f"{valor}: KPIs calculados para {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets "
          f"(dcor sobre submuestra de {len(idx_muestra)} filas).")

kpis_q2_14 = pd.DataFrame(resultados_q2_14)

BTCUSD: KPIs calculados para 63 variables x 2 targets (dcor sobre submuestra de 3000 filas).


### 11.1. Comparación con Pearson/Spearman: ¿aportan algo nuevo?

In [39]:
ranking_q2_14 = (
    kpis_q2_14
    .merge(ranking_q2_7[['valor', 'variable', 'target', 'pearson', 'spearman', 'coseno']],
           on=['valor', 'variable', 'target'], how='left')
    .assign(
        abs_pearson=lambda d: d['pearson'].abs(),
        dcor_sobre_pearson=lambda d: d['distance_corr'] - d['abs_pearson'],
        aporta_valor_dcor=lambda d: d['dcor_sobre_pearson'] > UMBRAL_DIFERENCIA_DCOR_Q2_14,
    )
)

aportes_q2_14 = []
for (valor, target_col), grupo in ranking_q2_14.groupby(['valor', 'target']):
    rank_pearson = grupo['abs_pearson'].rank(ascending=False, method='min')
    rank_mi = grupo['mutual_info'].rank(ascending=False, method='min')
    sube_en_mi = (rank_mi <= TOP_N_KPI_Q2_14) & (rank_pearson > TOP_N_KPI_Q2_14)
    for variable, en_top_mi in zip(grupo['variable'], sube_en_mi):
        aportes_q2_14.append({'valor': valor, 'target': target_col, 'variable': variable,
                               'entra_top_mi_sin_estar_en_pearson': bool(en_top_mi)})

ranking_q2_14 = (
    ranking_q2_14
    .merge(pd.DataFrame(aportes_q2_14), on=['valor', 'target', 'variable'], how='left')
    .drop(columns='abs_pearson')
    .sort_values(['valor', 'target', 'distance_corr'], ascending=[True, True, False])
    .reset_index(drop=True)
)

n_dcor_aporta = int(ranking_q2_14['aporta_valor_dcor'].sum())
n_mi_aporta = int(ranking_q2_14['entra_top_mi_sin_estar_en_pearson'].sum())
print(f"Variables donde dcor supera a |Pearson| en más de {UMBRAL_DIFERENCIA_DCOR_Q2_14}: {n_dcor_aporta}")
print(f"Variables que entran al top-{TOP_N_KPI_Q2_14} de información mutua sin estar en el top-{TOP_N_KPI_Q2_14} de |Pearson|: {n_mi_aporta}")
ranking_q2_14

Variables donde dcor supera a |Pearson| en más de 0.1: 20
Variables que entran al top-10 de información mutua sin estar en el top-10 de |Pearson|: 7


,valor,variable,target,n_mi,mutual_info,n_dcor,distance_corr,pearson,spearman,coseno,dcor_sobre_pearson,aporta_valor_dcor,entra_top_mi_sin_estar_en_pearson
0,BTCUSD,drawdown_historico,Close,40283,7.737850,3000,0.865113,0.866382,0.888647,-0.508247,-0.001268,False,False
1,BTCUSD,fundamental_edad,Close,40283,4.167230,3000,0.861998,0.792445,0.786920,0.950639,0.069553,False,False
2,BTCUSD,x3_atr_14,Close,40283,0.781033,3000,0.736225,0.674888,0.768740,0.914012,0.061337,False,False
3,BTCUSD,duracion_drawdown,Close,40283,3.374612,3000,0.721982,-0.659814,-0.748208,0.413424,0.062167,False,False
4,BTCUSD,drawdown_365d,Close,40283,6.377768,3000,0.550564,0.574979,0.568211,-0.503046,-0.024414,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,BTCUSD,pendiente_ema_26,retorno_log_1h,40257,0.000000,2999,0.048864,0.017293,0.007285,0.017298,0.031571,False,False
104,BTCUSD,pendiente_sma_20,retorno_log_1h,40244,0.005110,2999,0.047356,0.010009,0.000207,0.010015,0.037347,False,False
105,BTCUSD,drawdown_historico,retorno_log_1h,40282,0.027613,3000,0.045279,0.011014,0.014193,0.005599,0.034266,False,False
106,BTCUSD,pendiente_sma_50,retorno_log_1h,40184,0.003036,2993,0.044001,0.008213,0.005717,0.008221,0.035788,False,False


### 11.2. Guardar los KPIs y la comparación con Pearson/Spearman

Mismo criterio que las secciones anteriores. A diferencia del resto, este
CSV es deliberadamente independiente de la síntesis (`Q2-11`) y del export
principal (`Q2-5`) — ver nota de alcance en `docs/tracking/todos.md`
(`Q2-11`/`Q2-5` no listan `Q2-14` entre sus `[dep: ...]`): se guarda igual
para no perder el cálculo, pero queda como anexo exploratorio, no como
insumo obligatorio de los ítems de cierre.

In [40]:
for valor in tablas:
    path_kpis = CARPETA_EXP / f'{valor}_q2_14_kpis.csv'
    ranking_q2_14[ranking_q2_14['valor'] == valor].to_csv(path_kpis, index=False)
    print(f"{valor}: KPIs adicionales guardados en {path_kpis.name}")

BTCUSD: KPIs adicionales guardados en BTCUSD_q2_14_kpis.csv


## 12. Análisis por regímenes (Q2-13)

**Objetivo**: comparar cómo cambia la relación variable↔precio bajo distintos
regímenes del activo — tendencia (bull/bear/sideways), volatilidad
(alta/baja) y drawdown (sano/deteriorado) — para detectar relaciones que
solo valen en ciertos contextos de mercado.

**Pregunta**: la fuerza/dirección de la relación de cada variable con
`retorno_log_1h` (y `Close`, referencial) ¿es estable entre regímenes, o
depende fuertemente de en qué régimen está el activo?

**Metodología**: 3 dimensiones de régimen, cada una con categorías
definidas explícitamente (nunca "régimen" sin definir):
- **Tendencia**: `regimen_tendencia` (bull/bear/sideways), ya calculado en
  `X5_Q1` (`Q1-7`) a partir de `adx_14`/`x3_trend_slope_50`.
- **Volatilidad**: split por mediana de `x3_vol_24h` (alta si > mediana,
  baja si ≤ mediana) — mediana en vez de un umbral fijo porque la escala de
  `x3_vol_24h` es propia de cada activo.
- **Drawdown**: `drawdown_historico` contra un umbral fijo
  `UMBRAL_DRAWDOWN_DETERIORADO_Q2_13 = -0.20` (deteriorado si el activo está
  20% o más bajo su máximo histórico, sano si no).

Para cada dimensión/categoría: Pearson, Spearman y coseno por variable y
target (reutiliza `_pearson_spearman_coseno`, ya definida en `Q2-9`).
Clasificación de dependencia de régimen con el mismo criterio que
`Q2-9`/`Q2-10` (`UMBRAL_STD_INESTABLE`): cambia de signo entre categorías →
`cambio de signo`; no cambia de signo pero la dispersión entre categorías es
alta → `dependiente de régimen`; si no → `consistente`.

**Interpretación**: una relación "consistente" entre regímenes es más
confiable como insumo para `Q2-11`/`X5_Q3` que una que cambia de signo según
el contexto de mercado.

**Limitaciones**: con 1 solo activo (BTCUSD) y ~4.4 años de historia,
algunas categorías (ej. `sideways`, `deteriorado`) tienen menos
observaciones — se exige el mismo mínimo de 30 filas por celda que el resto
del notebook.

In [41]:
UMBRAL_DRAWDOWN_DETERIORADO_Q2_13 = -0.20  # drawdown_historico <= -20% desde el máximo histórico -> "deteriorado"

REGIMENES_Q2_13 = {
    'tendencia': 'regimen_tendencia',          # ya categórica: bull/bear/sideways (Q1-7)
    'volatilidad': 'volatilidad_regimen_q2_13',
    'drawdown': 'drawdown_regimen_q2_13',
}

for tabla in tablas.values():
    mediana_vol = tabla['x3_vol_24h'].median()
    tabla['volatilidad_regimen_q2_13'] = pd.Series(
        np.where(tabla['x3_vol_24h'] > mediana_vol, 'alta', 'baja'), index=tabla.index
    ).where(tabla['x3_vol_24h'].notna())
    tabla['drawdown_regimen_q2_13'] = pd.Series(
        np.where(tabla['drawdown_historico'] <= UMBRAL_DRAWDOWN_DETERIORADO_Q2_13, 'deteriorado', 'sano'), index=tabla.index
    ).where(tabla['drawdown_historico'].notna())

for valor, tabla in tablas.items():
    print(f"{valor} — volatilidad_regimen_q2_13 (mediana={mediana_vol:.4f}): "
          f"{tabla['volatilidad_regimen_q2_13'].value_counts(dropna=True).to_dict()}")
    print(f"{valor} — drawdown_regimen_q2_13 (umbral={UMBRAL_DRAWDOWN_DETERIORADO_Q2_13}): "
          f"{tabla['drawdown_regimen_q2_13'].value_counts(dropna=True).to_dict()}")


BTCUSD — volatilidad_regimen_q2_13 (mediana=0.4073): {'baja': 20130, 'alta': 20129}
BTCUSD — drawdown_regimen_q2_13 (umbral=-0.2): {'deteriorado': 26279, 'sano': 14004}


In [42]:
resultados_q2_13 = []
for valor, tabla in tablas.items():
    variables_analisis = [c for c in _variables_analisis_q2_3(tabla) if c not in REGIMENES_Q2_13.values()]
    for dim, col_regimen in REGIMENES_Q2_13.items():
        categorias = sorted(tabla[col_regimen].dropna().unique())
        for categoria in categorias:
            sub_regimen = tabla[tabla[col_regimen] == categoria]
            for target_col in TARGETS_Q2_3:
                for col in variables_analisis:
                    sub = sub_regimen[[col, target_col]].dropna()
                    metricas = _pearson_spearman_coseno(sub, col, target_col)
                    if metricas is None:
                        continue
                    resultados_q2_13.append({
                        'valor': valor, 'dimension_regimen': dim, 'categoria_regimen': categoria,
                        'variable': col, 'target': target_col, **metricas,
                    })

por_regimen_q2_13 = pd.DataFrame(resultados_q2_13)
print(f"Filas régimen x variable x target generadas: {len(por_regimen_q2_13)} (dimensiones: {list(REGIMENES_Q2_13)})")


Filas régimen x variable x target generadas: 756 (dimensiones: ['tendencia', 'volatilidad', 'drawdown'])


### 12.1. Clasificación de dependencia de régimen

In [43]:
resumen_q2_13 = []
for (valor, dim, variable, target_col), grupo in por_regimen_q2_13.groupby(['valor', 'dimension_regimen', 'variable', 'target']):
    pearsons = grupo['pearson'].to_numpy()
    signos = set(np.sign(pearsons)[np.sign(pearsons) != 0])
    if len(signos) > 1:
        clasificacion = 'cambio de signo'
    elif grupo['pearson'].std() > UMBRAL_STD_INESTABLE:
        clasificacion = 'dependiente de régimen'
    else:
        clasificacion = 'consistente'
    resumen_q2_13.append({
        'valor': valor, 'dimension_regimen': dim, 'variable': variable, 'target': target_col,
        'n_categorias': len(grupo), 'pearson_std_entre_categorias': grupo['pearson'].std(),
        'clasificacion': clasificacion,
    })

resumen_q2_13 = pd.DataFrame(resumen_q2_13)
print(resumen_q2_13.groupby('dimension_regimen')['clasificacion'].value_counts().to_string())

dims_dependientes_q2_13 = (
    resumen_q2_13[resumen_q2_13['clasificacion'] != 'consistente']
    .groupby(['valor', 'variable', 'target'])['dimension_regimen']
    .apply(lambda s: ','.join(sorted(s)))
    .rename('regimenes_con_dependencia').reset_index()
)
depende_de_regimen_q2_13 = (
    resumen_q2_13.assign(depende=lambda d: d['clasificacion'] != 'consistente')
    .groupby(['valor', 'variable', 'target'])['depende'].any()
    .rename('depende_de_algun_regimen').reset_index()
)
print(f"Pares variable/target con dependencia de al menos un régimen: "
      f"{int(depende_de_regimen_q2_13['depende_de_algun_regimen'].sum())}/{len(depende_de_regimen_q2_13)}")


dimension_regimen  clasificacion         
drawdown           consistente               61
                   cambio de signo           39
                   dependiente de régimen     8
tendencia          consistente               72
                   cambio de signo           36
volatilidad        consistente               86
                   cambio de signo           21
                   dependiente de régimen     1
Pares variable/target con dependencia de al menos un régimen: 68/108


### 12.2. Guardar el detalle por régimen y el resumen de dependencia

In [44]:
for valor in tablas:
    path_por_regimen = CARPETA_EXP / f'{valor}_q2_13_por_regimen.csv'
    path_resumen = CARPETA_EXP / f'{valor}_q2_13_resumen.csv'
    por_regimen_q2_13[por_regimen_q2_13['valor'] == valor].to_csv(path_por_regimen, index=False)
    resumen_q2_13[resumen_q2_13['valor'] == valor].to_csv(path_resumen, index=False)
    print(f"{valor}: detalle por régimen guardado en {path_por_regimen.name}, resumen en {path_resumen.name}")


BTCUSD: detalle por régimen guardado en BTCUSD_q2_13_por_regimen.csv, resumen en BTCUSD_q2_13_resumen.csv


## 13. Síntesis descriptiva y ranking consolidado (Q2-11)

**Objetivo**: consolidar en una sola tabla, por variable y target, las
métricas descriptivas ya calculadas (frecuencia efectiva, Pearson,
Spearman, coseno, no linealidad, estabilidad, régimen, redundancia) y
asignar un nivel de evidencia descriptiva explícito.

**Pregunta**: de las variables analizadas, ¿cuáles tienen evidencia
descriptiva sólida (relación con fuerza, estable en el tiempo, consistente
entre regímenes, no redundante) y cuáles no?

**Metodología**: join de `ranking_q2_7` (Pearson/Spearman/coseno/no
linealidad), `estabilidad_q2_9` (estabilidad temporal), el resumen de
`Q2-13` (dependencia de régimen), `pares_q2_12` (si la variable tiene algún
par redundante) y `cobertura_pct` del catálogo de `X5_Q1` (`Q1-9`) como
proxy de frecuencia efectiva — esta v2 no tiene un ítem dedicado de
"frecuencia efectiva de actualización" (sí lo hubo en v1/`X5_P2`), así que
se reutiliza la cobertura de datos real ya calculada en `Q1-9` en vez de
recalcularla.

**Nivel de evidencia descriptiva** (regla explícita, no un score de ML):
- `alta`: fuerza al menos "débil" por Pearson (`|pearson| >= 0.10`) **y**
  estable en el tiempo (`Q2-9`) **y** consistente entre los 3 regímenes
  (`Q2-13`).
- `media`: tiene fuerza pero falla en estabilidad temporal **o** en régimen
  (no en ambas a la vez).
- `baja`: sin fuerza (`|pearson| < 0.10`, incluida "nula/despreciable"), o
  con fuerza pero inestable en el tiempo **y** dependiente de régimen a la
  vez.

**Interpretación**: esta tabla es el insumo directo de `Q2-5` (export) y de
`X5_Q3` (hipótesis parámetro↔variable). **No es un ranking predictivo**:
evidencia descriptiva sólida no garantiza poder predictivo fuera de
muestra — eso se valida con backtesting en una etapa posterior, fuera de
alcance de `X5_alt_v2`.

**Limitaciones**: hereda las limitaciones de cada sección que consolida —
las columnas `x2_*`/`fundamental_*` (100% nulas en este snapshot de Mac) ya
quedaron fuera de `ranking_q2_7` desde `Q2-3` (sin datos suficientes para
Pearson), así que tampoco aparecen acá.

In [45]:
_cobertura_partes = []
for valor in tablas:
    catalogo_valor = pd.read_csv(carpeta_exp('X5_Q1') / f'{valor}_catalogo_variables.csv')[['variable', 'cobertura_pct']]
    catalogo_valor['valor'] = valor
    _cobertura_partes.append(catalogo_valor)
cobertura_q2_11 = pd.concat(_cobertura_partes, ignore_index=True)

_redundancia_por_variable_q2_11 = {}
for valor in tablas:
    sub = pares_q2_12[(pares_q2_12['valor'] == valor) & pares_q2_12['redundante']]
    _redundancia_por_variable_q2_11[valor] = set(sub['var_a']) | set(sub['var_b'])

sintesis_q2_11 = (
    ranking_q2_7
    .merge(estabilidad_q2_9[['valor', 'variable', 'target', 'estabilidad', 'pearson_std_entre_anios']],
           on=['valor', 'variable', 'target'], how='left')
    .merge(depende_de_regimen_q2_13, on=['valor', 'variable', 'target'], how='left')
    .merge(dims_dependientes_q2_13, on=['valor', 'variable', 'target'], how='left')
    .merge(cobertura_q2_11, on=['valor', 'variable'], how='left')
)
sintesis_q2_11['depende_de_algun_regimen'] = sintesis_q2_11['depende_de_algun_regimen'].fillna(False)
sintesis_q2_11['regimenes_con_dependencia'] = sintesis_q2_11['regimenes_con_dependencia'].fillna('')
sintesis_q2_11['tiene_par_redundante'] = sintesis_q2_11.apply(
    lambda r: r['variable'] in _redundancia_por_variable_q2_11.get(r['valor'], set()), axis=1)


def _nivel_evidencia_q2_11(row):
    fuerza_ok = pd.notna(row['pearson']) and abs(row['pearson']) >= 0.10
    if not fuerza_ok:
        return 'baja'
    estable_ok = row['estabilidad'] == 'estable'
    regimen_ok = not row['depende_de_algun_regimen']
    if estable_ok and regimen_ok:
        return 'alta'
    if estable_ok or regimen_ok:
        return 'media'
    return 'baja'


sintesis_q2_11['nivel_evidencia_descriptiva'] = sintesis_q2_11.apply(_nivel_evidencia_q2_11, axis=1)

_ORDEN_NIVEL_Q2_11 = {'alta': 0, 'media': 1, 'baja': 2}
sintesis_q2_11 = (
    sintesis_q2_11
    .assign(_orden_nivel=lambda d: d['nivel_evidencia_descriptiva'].map(_ORDEN_NIVEL_Q2_11))
    .sort_values(['valor', 'target', '_orden_nivel'])
    .drop(columns='_orden_nivel')
    .reset_index(drop=True)
)

print(sintesis_q2_11.groupby('target')['nivel_evidencia_descriptiva'].value_counts().to_string())


target          nivel_evidencia_descriptiva
Close           baja                           47
                media                           6
                alta                            1
retorno_log_1h  baja                           35
                alta                           19


### 13.1. Guardar la síntesis consolidada

In [46]:
for valor in tablas:
    path_sintesis = CARPETA_EXP / f'{valor}_q2_11_sintesis.csv'
    sintesis_q2_11[sintesis_q2_11['valor'] == valor].to_csv(path_sintesis, index=False)
    print(f"{valor}: síntesis consolidada guardada en {path_sintesis.name}")


BTCUSD: síntesis consolidada guardada en BTCUSD_q2_11_sintesis.csv


## 14. Export estructurado de métricas por variable (Q2-5)

**Objetivo**: generar, por variable, un archivo estructurado (JSON —
formato natural para datos anidados, a diferencia de las tablas planas en
CSV ya guardadas en cada sección) con Pearson, Spearman, coseno, no
linealidad, ceteris paribus, retorno pasado, estabilidad, régimen y
redundancia — pensado para alimentar después un análisis por IA que
concluya cosas como "el drawdown se relaciona con N" (puente directo hacia
`X5_Q3`).

**Metodología**: no recalcula nada — recopila en un único dict anidado
`{variable: {...}}` las métricas ya calculadas y guardadas en CSV por
`Q2-3`/`Q2-4`/`Q2-6`/`Q2-7`/`Q2-8`/`Q2-9`/`Q2-11`/`Q2-12`/`Q2-13` (en
memoria, vía las tablas `ranking_q2_7`, `ranking_q2_6`, `ranking_q2_8`,
`estabilidad_q2_9`, `sintesis_q2_11`, `pares_q2_12`, `resumen_q2_13` — todas
ya construidas más arriba en este mismo notebook). Un archivo por activo:
`{valor}_q2_5_export_metricas.json`.

**Interpretación**: es el mismo universo de variables que `Q2-11`
(`ranking_q2_7['variable'].unique()`) — las excluidas allí por falta de
datos (`x2_*`/`fundamental_*` en este snapshot) tampoco aparecen acá.

**Limitaciones**: estructura pensada para lectura programática, no para
inspección visual en el notebook (para eso están los CSV/`sintesis_q2_11`
de las secciones anteriores).

In [47]:
import json


def _limpiar_para_json(valor):
    """NaN/NaT -> None y tipos numpy -> tipos nativos de Python, para que json.dump no falle."""
    if valor is None:
        return None
    if isinstance(valor, (np.floating, float)) and np.isnan(valor):
        return None
    if pd.isna(valor):
        return None
    if isinstance(valor, np.integer):
        return int(valor)
    if isinstance(valor, np.floating):
        return float(valor)
    if isinstance(valor, np.bool_):
        return bool(valor)
    return valor


def _limpiar_fila(fila):
    return {k: _limpiar_para_json(v) for k, v in fila.items()}


In [48]:
export_q2_5 = {}
for valor in tablas:
    variables_valor = sorted(ranking_q2_7.loc[ranking_q2_7['valor'] == valor, 'variable'].unique())
    export_valor = {}

    for variable in variables_valor:
        por_target = {}
        for target_col in TARGETS_Q2_3:
            fila_clasica = ranking_q2_7[(ranking_q2_7['valor'] == valor) & (ranking_q2_7['variable'] == variable)
                                         & (ranking_q2_7['target'] == target_col)]
            if fila_clasica.empty:
                continue
            f = fila_clasica.iloc[0]
            bloque = {
                'pearson': _limpiar_para_json(f['pearson']),
                'spearman': _limpiar_para_json(f['spearman']),
                'coseno': _limpiar_para_json(f['coseno']),
                'direccion_fuerza': f['direccion_fuerza'],
                'no_linealidad': {
                    'r2_lin': _limpiar_para_json(f['r2_lin']), 'r2_cuad': _limpiar_para_json(f['r2_cuad']),
                    'delta_r2': _limpiar_para_json(f['delta_r2']), 'forma': f['forma'],
                },
            }
            fila_est = estabilidad_q2_9[(estabilidad_q2_9['valor'] == valor) & (estabilidad_q2_9['variable'] == variable)
                                         & (estabilidad_q2_9['target'] == target_col)]
            if not fila_est.empty:
                e = fila_est.iloc[0]
                bloque['estabilidad'] = {
                    'clasificacion': e['estabilidad'],
                    'pearson_std_entre_anios': _limpiar_para_json(e['pearson_std_entre_anios']),
                }
            fila_sint = sintesis_q2_11[(sintesis_q2_11['valor'] == valor) & (sintesis_q2_11['variable'] == variable)
                                        & (sintesis_q2_11['target'] == target_col)]
            if not fila_sint.empty:
                s = fila_sint.iloc[0]
                bloque['nivel_evidencia_descriptiva'] = s['nivel_evidencia_descriptiva']
                bloque['cobertura_pct'] = _limpiar_para_json(s['cobertura_pct'])
            por_target[target_col] = bloque

        ceteris_var = ranking_q2_6[(ranking_q2_6['valor'] == valor) & (ranking_q2_6['variable'] == variable)]
        ceteris_bloque = {
            fila['target']: {
                'coef_ceteris_paribus': _limpiar_para_json(fila['coef_ceteris_paribus']),
                'vif': _limpiar_para_json(fila['vif']),
                'coincide_signo_con_pearson': _limpiar_para_json(fila['coincide_signo_con_pearson']),
            }
            for fila in ceteris_var.to_dict('records')
        }

        retorno_pasado_var = ranking_q2_8[(ranking_q2_8['valor'] == valor) & (ranking_q2_8['variable'] == variable)]
        retorno_pasado_bloque = {
            f"{int(fila['horizonte_h'])}h": {
                'pearson': _limpiar_para_json(fila['pearson']), 'spearman': _limpiar_para_json(fila['spearman']),
            }
            for fila in retorno_pasado_var.to_dict('records')
        }

        regimen_var = resumen_q2_13[(resumen_q2_13['valor'] == valor) & (resumen_q2_13['variable'] == variable)]
        regimen_bloque = {
            fila['dimension_regimen']: {
                'clasificacion': fila['clasificacion'],
                'pearson_std_entre_categorias': _limpiar_para_json(fila['pearson_std_entre_categorias']),
            }
            for fila in regimen_var.to_dict('records')
        }

        redundancia_var = pares_q2_12[(pares_q2_12['valor'] == valor)
                                       & ((pares_q2_12['var_a'] == variable) | (pares_q2_12['var_b'] == variable))
                                       & pares_q2_12['redundante']]
        pares_redundantes_lista = sorted((set(redundancia_var['var_a']) | set(redundancia_var['var_b'])) - {variable})

        export_valor[variable] = {
            'targets': por_target,
            'ceteris_paribus': ceteris_bloque,
            'retorno_pasado': retorno_pasado_bloque,
            'regimen': regimen_bloque,
            'redundancia': {
                'tiene_par_redundante': len(pares_redundantes_lista) > 0,
                'pares_redundantes': pares_redundantes_lista,
            },
        }
    export_q2_5[valor] = export_valor

print(f"Export estructurado construido para {sum(len(v) for v in export_q2_5.values())} variables en total "
      f"({', '.join(f'{v}: {len(d)}' for v, d in export_q2_5.items())}).")


Export estructurado construido para 54 variables en total (BTCUSD: 54).


### 14.1. Guardar el export estructurado

In [49]:
for valor in tablas:
    path_json = CARPETA_EXP / f'{valor}_q2_5_export_metricas.json'
    with open(path_json, 'w', encoding='utf-8') as f:
        json.dump(export_q2_5[valor], f, ensure_ascii=False, indent=2)
    print(f"{valor}: export estructurado de métricas guardado en {path_json.name} ({len(export_q2_5[valor])} variables)")


BTCUSD: export estructurado de métricas guardado en BTCUSD_q2_5_export_metricas.json (54 variables)


## 15. Exportar resultados a zip (Windows → Mac)

Este notebook corre en Windows con `Data/`/`resources/x2/`/`resources/x3/`
actualizados (no disponibles así en Mac, ver `CLAUDE.md`). Todo lo generado
en las Secciones 3-14 (métricas, rankings, export estructurado — nunca los
gráficos, ver convención en `docs/plans/X5_alternativo.md` sección 10.2) se
escribe directamente en `resources/x5_alt/exp_Q2/` (limpiada al inicio por
`limpiar_exp`, Sección 0). La última celda comprime esa carpeta completa en
`resources/x5_alt/exp_Q2.zip` con `exportar_zip` (`scripts/x5_alt_export.py`,
mismo módulo que usa `X5_Q1.ipynb`).

In [50]:
exportar_zip('X5_Q2')


X5_Q2: zip con 21 archivo(s) guardado en /Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/X5_Q2_20261010_230303.zip


PosixPath('/Users/macasaez/Desktop/claude_projects_v2/Alginvesting/resources/x5_alt/X5_Q2_20261010_230303.zip')